In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
names = ["000TJ_T", "001DS_D", "002ED_E", "003TL_T", "004TR_T", "005JQ_J", "006WB_W", "007AM_A", "008SL_S", "009EP_E", "010MR_M", "011BM_B", "012EM_E", "013ZM_Z", "014ZG_Z", "015AL_A", "016MP_M", "017PB_P", "018TL_T", "019WU_W", "020CD_C", "021AD_A", "022SG_S", "023MT_M", "024JG_J", "025AA_A", "026ED_E", "027IJ_I", "028PL_P", "029RT_R"]
results_folder = ".\\Results\\"
timestamps_folder = ".\\Timestamps\\"

In [ ]:
# Cette fonction lit les timestamps de référence pour une tâche donnée.
def read_reference_timestamps(task_name):
    with open(f"{timestamps_folder}{task_name.split('.')[0]}.txt", 'r') as file:
        return [float(line.strip().replace(',', '.')) for line in file]

#Cette fonction calcule la différence entre les timing d'appuis
def calculate_absolute_errors(user_timestamps, reference_timestamps):
    return np.abs(user_timestamps - reference_timestamps)

#Cette fonction calcule les erreur d'interval. C'est à dire qu'elle compare les zones entre les appuis et non pas les timing entre les appuis.
def calculate_interval_errors(user_timestamps, reference_timestamps):
    ref_diffs = np.diff(reference_timestamps)
    user_diffs = np.diff(user_timestamps)

    percentage_diffs = (user_diffs / ref_diffs)
    absolute_percentage_errors = np.abs(1 - percentage_diffs)

    return absolute_percentage_errors


# Cette fonction normalise les valeurs entre 0 et 100 (forme de pourcentage). La première valeur est mise à 0 et la dernière 100. Les autres sont ensuite misent à l'echelle.
def calculate_normalized_segments(timestamps):
    normalized_timestamps = (np.array(timestamps) - timestamps[0]) / (timestamps[-1] - timestamps[0]) * 100
    segment_percentages = np.diff(normalized_timestamps)
    return segment_percentages

#Cette fonction compare les taux de pourcentage les segment prennent entre les timestamps. Cette mesure est interessante car elle mesure réellement si le user était en rythme.
def calculate_mean_percentage_error(user_timestamps, reference_timestamps):
    user_segments = calculate_normalized_segments(user_timestamps)
    reference_segments = calculate_normalized_segments(reference_timestamps)
    min_length = min(len(user_segments), len(reference_segments))

    user_segments = user_segments[:min_length]
    reference_segments = reference_segments[:min_length]

    differences = np.abs(user_segments - reference_segments)
    mean_error = np.mean(differences)    
    return mean_error

# Calcule l'erreur quadratique moyenne entre les intervals des deux séries.
def mse_intervals(user_timestamps, reference_timestamps):
    user_intervals = np.diff(user_timestamps)
    reference_intervals = np.diff(reference_timestamps)
    mse = np.mean((user_intervals - reference_intervals) ** 2)
    return mse


In [ ]:
def calculate_errors(user_timestamps, reference_timestamps, monophasic = True):
    reference_timestamps = reference_timestamps[:-1]
    user_timestamps[0] = 0

    user_timestamps = np.array(user_timestamps)
    reference_timestamps = np.array(reference_timestamps)
    
    if monophasic:
        user_timestamps = user_timestamps[::2]
        reference_timestamps = reference_timestamps[::2]
    
    if len(user_timestamps) != len(reference_timestamps):
        min_length = min(len(user_timestamps), len(reference_timestamps))
        user_timestamps = user_timestamps[:min_length]
        reference_timestamps = reference_timestamps[:min_length]

    # Calcul des erreurs absolues
    absolute_errors = calculate_absolute_errors(user_timestamps, reference_timestamps)
    
    # Calcul des erreurs relatives
    relative_errors = calculate_interval_errors(user_timestamps, reference_timestamps)

    #calcul des ecart des taux de pourcentage
    percentage_difference_error = calculate_mean_percentage_error(user_timestamps, reference_timestamps)

    # Calcul test
    mse_error = mse_intervals(user_timestamps, reference_timestamps)

    return absolute_errors, relative_errors, percentage_difference_error, mse_error

In [ ]:
resultsByPatient = {}

for name in names:
    resultsByPatient[name] = []
    patient_folder = f"{results_folder}{name}\\"
    for csv_file in os.listdir(patient_folder):
        if csv_file.endswith(name + ".csv"):
            df = pd.read_csv(f"{patient_folder}{csv_file}", sep=';', decimal=',', skiprows=2)
            for task in df.columns:
                if task and 'Unnamed' not in task and 'ReflexTest' not in task and 'Button' not in task:
                    user_timestamps = df[task].dropna().tolist()
                    reference_timestamps = read_reference_timestamps(task)
                    abs_errors, int_errors, percentage_diff_error, mse_error = calculate_errors(user_timestamps, reference_timestamps)

                    mean_abs_error = np.mean(abs_errors)
                    mean_int_error = np.mean(int_errors)
                    mean_prc_error = np.mean(percentage_diff_error)
                    mean_mse_error = np.mean(mse_error)

                    resultsByPatient[name].append(np.array([task.split('.')[0], percentage_diff_error]))
    resultsByPatient[name].sort(key=lambda x: x[0])



In [ ]:
def save_results(resultsByPatient, output_filename):
    # Initialisation du DataFrame avec la ligne d'en-tête
    # On suppose que le dictionnaire n'est pas vide et que tous les patients ont un ensemble identique de tâches
    first_patient_name = next(iter(resultsByPatient))  # Obtenir le nom du premier patient
    headers = ["Nom"] + [task[0] for task in resultsByPatient[first_patient_name]]  # Création des en-têtes
    
    # Création d'un DataFrame vide avec les en-têtes spécifiés
    df = pd.DataFrame(columns=headers)
    
    # Remplissage du DataFrame avec les données pour chaque patient
    for patient_name, tasks in resultsByPatient.items():
        # Initialisation de la ligne pour le patient actuel
        patient_data = [patient_name]  # Le nom du patient est la première donnée
        
        # Ajout des données de chaque tâche pour ce patient
        for task_data in tasks:
            # On suppose que task_data[1] contient la donnée à ajouter; ajustez selon votre structure
            patient_data.append(task_data[1])
        
        # Ajout de la ligne du patient actuel au DataFrame
        df.loc[len(df)] = patient_data  # Ajoute une nouvelle ligne au DataFrame avec les données du patient
    
    # Sauvegarde du DataFrame dans un fichier Excel
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        df.to_excel(writer, index=False)
    
    print(f"File saved as {output_filename}")


save_results(resultsByPatient, "DataRK2Fiability_ProportionnalIntervals.xlsx")


In [ ]:
def load_results(input_filename):
    # Lecture du fichier Excel
    df = pd.read_excel(input_filename)

    # Initialisation du dictionnaire pour stocker les données rechargées
    resultsByPatient_reloaded = {}

    # Itération sur chaque ligne du DataFrame pour reconstruire le dictionnaire
    for index, row in df.iterrows():
        patient_name = row['Nom']  # Le nom du patient se trouve dans la colonne 'Nom'
        tasks_data = row[1:].to_dict()  # Conversion des données de tâches en dictionnaire
        
        # Initialisation de la liste des tâches pour ce patient
        tasks_list = []

        for task, data in tasks_data.items():
            if pd.notna(data):
                # Ignorer les suffixes numériques pour regrouper les tâches similaires
                task_name = task.split('.')[0]  # Suppression du suffixe numérique
                tasks_list.append((task_name, data))
        
        # Regroupement des données pour les tâches avec le même nom
        tasks_grouped = {}
        for task_name, data in tasks_list:
            if task_name in tasks_grouped:
                # Si la tâche existe déjà, on ajoute la donnée à la liste existante
                tasks_grouped[task_name].append(data)
            else:
                # Sinon, on crée une nouvelle liste pour cette tâche
                tasks_grouped[task_name] = [data]

        # Pour simplifier, on prend la moyenne si plusieurs données existent pour la même tâche
        # Vous pouvez ajuster cette logique selon vos besoins
        tasks_final = [(task_name, sum(values)/len(values)) for task_name, values in tasks_grouped.items()]

        # Ajout des données du patient au dictionnaire
        resultsByPatient_reloaded[patient_name] = tasks_final

    return resultsByPatient_reloaded

resultsByPatient_mean = load_results("DataRK_ProportionnalIntervals.xlsx")

print(resultsByPatient_mean)



In [ ]:
#save_results(resultsByPatient_mean, "DataRK_ProportionnalIntervalsMean.xlsx")

In [ ]:
def remove_outliers_and_replace_with_median(data_dict):
    # Regroup scores by task
    task_scores = {}
    for patient_scores in data_dict.values():
        for task, score in patient_scores:
            if task not in task_scores:
                task_scores[task] = []
            task_scores[task].append(score)
            
    # Calculate medians and bounds for outliers for each task
    task_medians = {}
    task_bounds = {}
    for task, scores in task_scores.items():
        q1, q3 = np.percentile(scores, [25, 75])
        iqr = q3 - q1
        median = np.median(scores)
        task_bounds[task] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
        task_medians[task] = median
    
    # Replace outliers with median
    cleaned_data = {}
    for patient, patient_scores in data_dict.items():
        cleaned_scores = []
        for task, score in patient_scores:
            lower_bound, upper_bound = task_bounds[task]
            if lower_bound <= score <= upper_bound:
                cleaned_scores.append((task, score))
            else: 
                cleaned_scores.append((task, task_medians[task]))
        cleaned_data[patient] = cleaned_scores

    return cleaned_data


outliers_resultsByPatient_mean = remove_outliers_and_replace_with_median(resultsByPatient_mean)
print(outliers_resultsByPatient_mean)
#save_results(outliers_resultsByPatient_mean, "DataRK_ProportionnalIntervalsMeanOutliers.xlsx")


In [ ]:
import pandas as pd
from statsmodels.formula.api import ols
import statsmodels.api as sm

# Transformation des données en une liste de dictionnaires pour faciliter la création du DataFrame
data_list = []

for subject, tasks in outliers_resultsByPatient_mean.items():
    for task in tasks:
        song, condition_feedback = task[0].split('_')
        condition = condition_feedback[:-1] if condition_feedback[-1] in ['N'] else condition_feedback
        feedback = 'No' if condition_feedback.endswith('N') else 'Yes'
        data_list.append({'Subject': subject, 'Song': song, 'Condition': condition, 'Feedback': feedback, 'Score': task[1]})

# Création du DataFrame
df = pd.DataFrame(data_list)



In [ ]:
import pandas as pd
import os

# Supposons que vous ayez déjà votre dataframe df comme indiqué dans votre question.

# Emplacement du dossier contenant les résultats des sujets
results_folder = "./Results/"

# Liste des noms de sujets
names = ["000TJ_T", "001DS_D", "002ED_E", "003TL_T", "004TR_T", "005JQ_J", "006WB_W", "007AM_A", "008SL_S", "009EP_E", "010MR_M", "011BM_B", "012EM_E", "013ZM_Z", "014ZG_Z", "015AL_A", "016MP_M", "017PB_P", "018TL_T", "019WU_W", "020CD_C", "021AD_A", "022SG_S", "023MT_M", "024JG_J", "025AA_A", "026ED_E", "027IJ_I", "028PL_P", "029RT_R"]

# Structure pour stocker les données des participants
participants_info = {}

for name in names:
    patient_folder = f"{results_folder}{name}/"
    for csv_file in os.listdir(patient_folder):
        if csv_file.endswith(name + ".csv"):
            df_subject = pd.read_csv(f"{patient_folder}{csv_file}", sep=';', decimal=',', nrows=1)
            # Récupération des données spécifiques pour le musicien et le joueur
            musician = df_subject['Musicien'].iloc[0]
            music_years = df_subject['Nb_Annees'].iloc[0]
            gamer_info = df_subject['Jeux_videos'].iloc[0]
            participants_info[name] = {'Musician': musician, 'Musician_years': music_years, 'Gamer': gamer_info}

# Ajout des informations à la dataframe df
df['Musician'] = df['Subject'].apply(lambda x: participants_info[x]['Musician'])
df['Musician_years'] = df['Subject'].apply(lambda x: participants_info[x]['Musician_years'])
df['Gamer'] = df['Subject'].apply(lambda x: participants_info[x]['Gamer'])

# Affichage des premières lignes pour vérifier
print(df.head())


In [ ]:
# ============================================================
# Reaction‑time & tapping processing  +  fatigue analysis
# ============================================================
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.anova import AnovaRM
from scipy.stats import friedmanchisquare
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# ❶  Containers
# ------------------------------------------------------------------
resultsByPatient = {}            # tapping errors etc.
rtByPatient      = {}            # raw RT values by moment per patient

# helper – map 0‑based column index ➜ moment label
def _moment_from_position(col_idx):
    if   col_idx in (1, 2, 3):            return 'T0'
    elif col_idx in (52, 53, 54):         return 'T1'
    elif col_idx in (103, 104, 105):      return 'T2'
    elif col_idx in (154, 155, 156):      return 'T3'
    else:                                 return None

# ------------------------------------------------------------------
# ❷  Iterate through patients and their CSVs
# ------------------------------------------------------------------
for name in names:
    resultsByPatient[name] = []
    rtByPatient[name]      = {'T0': [], 'T1': [], 'T2': [], 'T3': []}

    patient_folder = f"{results_folder}{name}\\"
    for csv_file in os.listdir(patient_folder):
        if csv_file.endswith(f"{name}.csv"):
            df = pd.read_csv(f"{patient_folder}{csv_file}",
                             sep=';', decimal=',', skiprows=2)

            # scan columns with numeric index
            for col_idx, task in enumerate(df.columns):
                if not task or 'Unnamed' in task or 'Button' in task:
                    continue

                # ---- A)  ReflexTest iterations ---------------------------
                if 'ReflexTest' in task:
                    rt_vals = pd.to_numeric(df[task], errors='coerce').dropna()
                    moment = _moment_from_position(col_idx)
                    if moment and not rt_vals.empty:
                        rtByPatient[name][moment].extend(rt_vals.tolist())
                    continue   # skip to next column

                # ---- B)  All other (tapping) tasks -----------------------
                user_ts = df[task].dropna().tolist()
                ref_ts  = read_reference_timestamps(task)

                abs_err, int_err, prc_err, mse_err = calculate_errors(
                    user_ts, ref_ts
                )
                resultsByPatient[name].append(
                    np.array([task.split('.')[0], prc_err])
                )

    # keep a stable order for later plotting
    resultsByPatient[name].sort(key=lambda x: x[0])

print(f"✔  Parsed tapping tasks for {len(resultsByPatient)} patients.")
print(f"✔  Parsed reaction‑time tasks for the same set of patients.")

# ------------------------------------------------------------------
# ❸  Build Subject × Moment table  (mean of the 3 iterations)
# ------------------------------------------------------------------
rows = []
for subj, moments in rtByPatient.items():
    row = {'Subject': subj}
    for moment in ['T0', 'T1', 'T2', 'T3']:
        vals = moments[moment]
        row[f'RT_{moment}'] = np.nan if len(vals) == 0 else np.nanmean(vals)
    rows.append(row)

rt_subject = pd.DataFrame(rows)

# ------------------------------------------------------------------
# ❹  Split complete vs incomplete cases
# ------------------------------------------------------------------
complete_mask = rt_subject[['RT_T0', 'RT_T1', 'RT_T2', 'RT_T3']].notna().all(axis=1)
rt_complete   = rt_subject[complete_mask].copy()

print(f"✓  Complete subjects : {rt_complete.shape[0]}")
print(f"✗  Incomplete        : {rt_subject.shape[0] - rt_complete.shape[0]}")

# ------------------------------------------------------------------
# ❺  Repeated‑measures ANOVA (complete cases)
# ------------------------------------------------------------------
rt_long_c = rt_complete.melt(id_vars='Subject',
                             value_vars=['RT_T0', 'RT_T1', 'RT_T2', 'RT_T3'],
                             var_name='Time', value_name='RT_ms')
rt_long_c['Time'] = rt_long_c['Time'].str.replace('RT_', '')

aov_rt = AnovaRM(rt_long_c, depvar='RT_ms',
                 subject='Subject', within=['Time']).fit()

print("\n=== RM‑ANOVA (complete cases) ===")
print(aov_rt.summary())

# ------------------------------------------------------------------
# ❻  Friedman non‑parametric test (all subjects)
# ------------------------------------------------------------------
friedman_data = rt_subject[['RT_T0', 'RT_T1', 'RT_T2', 'RT_T3']].dropna()
stat_f, p_f   = friedmanchisquare(*[friedman_data[c] for c in friedman_data.columns])
print(f"\nFriedman χ²(3) = {stat_f:.2f},  p = {p_f:.4f}")

# ------------------------------------------------------------------
# ❼  Pairwise comparisons (Bonferroni‑corrected, complete cases)
# ------------------------------------------------------------------
try:
    import pingouin as pg
    pairwise = pg.pairwise_ttests(
    dv='RT_ms', within='Time', subject='Subject',
    padjust='bonf', data=rt_long_c, effsize='hedges'  # ← change ici
)

    print("\n=== Pairwise (Bonferroni) ===")
    print(pairwise[['A','B','T','dof','p-corr','hedges']].round(3))
except ImportError:
    print("\n(Install Pingouin for automatic pairwise tests)")

# ------------------------------------------------------------------
# ❽  Descriptive stats  +  plot
# ------------------------------------------------------------------
desc = (
    rt_long_c.groupby('Time')['RT_ms']
             .agg(['mean', 'sem'])
             .rename(columns={'mean':'Mean_ms','sem':'SEM_ms'})
             .round(2)
)
print("\n=== Descriptives (Mean ± SEM, complete cases) ===")
print(desc)

plt.figure(figsize=(5,4), facecolor='white')
plt.errorbar(desc.index, desc['Mean_ms'], yerr=desc['SEM_ms'],
             marker='o', linewidth=2, capsize=5)
plt.ylabel('Reaction time (ms)')
plt.title('RT across testing session (complete subjects)')
plt.tight_layout()
plt.show()
# plt.savefig('RT_over_time.png', dpi=300)


In [ ]:
# ============================================================
#   RT analysis – MEAN of the three iterations (T0 … T3)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.stats.anova import AnovaRM
from scipy.stats import friedmanchisquare

# -------- build subject × moment table ----------------------
rt_rows = []
for subj, d in rtByPatient.items():
    rt_rows.append({'Subject': subj,
                    'RT_T0': np.nanmean(d['T0']) if d['T0'] else np.nan,
                    'RT_T1': np.nanmean(d['T1']) if d['T1'] else np.nan,
                    'RT_T2': np.nanmean(d['T2']) if d['T2'] else np.nan,
                    'RT_T3': np.nanmean(d['T3']) if d['T3'] else np.nan})
rt_subj = pd.DataFrame(rt_rows)

# -------- ANOVA (complete cases) ----------------------------
complete = rt_subj.dropna()
rt_long  = complete.melt(id_vars='Subject', var_name='Time', value_name='RT')
rt_long['Time'] = rt_long['Time'].str[-2:]   # keep T0 … T3

anova = AnovaRM(rt_long, 'RT', 'Subject', within=['Time']).fit()
print(anova)

# -------- Friedman on all available -------------------------
fr_data = rt_subj[['RT_T0','RT_T1','RT_T2','RT_T3']].dropna()
print('Friedman:', friedmanchisquare(*[fr_data[c] for c in fr_data]))

# -------- Plot ---------------------------------------------
desc = rt_long.groupby('Time')['RT'].agg(['mean','sem'])
plt.errorbar(desc.index, desc['mean'], yerr=desc['sem'], marker='o', capsize=5)
plt.ylabel('Reaction time (ms)'); plt.title('Reaction time over the study')
plt.show()


In [ ]:
# ============================================================
# Reaction‑time analysis with full sample
# ============================================================
import numpy as np, pandas as pd
from scipy import stats
import itertools

# ------------------------------------------------------------
# 1)  Subject × Moment table  (mean & median, keep NaNs)
# ------------------------------------------------------------
rows = []
for subj, d in rtByPatient.items():
    row = {'Subject': subj}
    for moment in ['T0', 'T1', 'T2', 'T3']:
        clean_vals = [v for v in d[moment] if not np.isnan(v)]
        row[f'{moment}_mean']   = np.nan if not clean_vals else np.mean(clean_vals)
        row[f'{moment}_median'] = np.nan if not clean_vals else np.median(clean_vals)
    rows.append(row)

rt_subj = pd.DataFrame(rows)

# ------------------------------------------------------------
# 2)  Descriptives with ranges
# ------------------------------------------------------------
desc_rows = []
for moment in ['T0','T1','T2','T3']:
    mvals = rt_subj[f'{moment}_mean']
    valid = mvals.dropna()
    desc_rows.append({
        'Moment' : moment,
        'N subj' : valid.size,
        'Mean'   : round(valid.mean(), 2),
        'Median' : round(valid.median(), 2),
        'Min'    : round(valid.min(), 2),
        'Max'    : round(valid.max(), 2)
    })

desc_df = pd.DataFrame(desc_rows)
print("=== Descriptives (mean of 3 iterations) ===")
print(desc_df.to_string(index=False))

# ------------------------------------------------------------
# 3)  Pairwise comparisons  (mean values)
# ------------------------------------------------------------
pairs = list(itertools.combinations(['T0','T1','T2','T3'], 2))
alpha  = 0.05 / len(pairs)   # Bonferroni

print(f"\n=== Pairwise tests (Bonferroni α={alpha:.3f}) ===")
for a, b in pairs:
    x = rt_subj[f'{a}_mean']
    y = rt_subj[f'{b}_mean']
    both = ~(x.isna() | y.isna())
    xa, ya = x[both], y[both]

    if both.sum() < 5:
        print(f"{a} vs {b}:   too few pairs (n={both.sum()}) — skipped")
        continue

    if both.sum() >= 15:
        stat, p = stats.ttest_rel(xa, ya, nan_policy='omit')
        test = "t"
        df   = both.sum() - 1
    else:
        stat, p = stats.wilcoxon(xa, ya)
        test = "W"
        df   = both.sum()

    signif = "*" if p < alpha else "n.s."
    print(f"{a} vs {b}:   {test}={stat:.2f}, df={df}, p={p:.4f}  → {signif}")

# ------------------------------------------------------------
# 4)  Optional — same workflow with MEDIANS ------------------
do_median = False   # ← passe à True si tu veux répéter l’analyse

if do_median:
    print("\n=== Running the same with MEDIAN per subject ===")
    desc_rows_m = []
    for moment in ['T0','T1','T2','T3']:
        mvals = rt_subj[f'{moment}_median']
        valid = mvals.dropna()
        desc_rows_m.append({
            'Moment': moment,
            'N subj': valid.size,
            'Median': round(valid.median(),2),
            'Min'   : round(valid.min(),2),
            'Max'   : round(valid.max(),2)
        })
    print(pd.DataFrame(desc_rows_m).to_string(index=False))

    print("\nPairwise Wilcoxon on medians:")
    for a, b in pairs:
        x = rt_subj[f'{a}_median']
        y = rt_subj[f'{b}_median']
        both = ~(x.isna() | y.isna())
        if both.sum() >= 5:
            stat, p = stats.wilcoxon(x[both], y[both])
            signif = "*" if p < alpha else "n.s."
            print(f"{a} vs {b}:   W={stat:.1f}, n={both.sum()}, p={p:.4f} → {signif}")


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure rt_subj is in scope; recompute descriptives if needed
moments = ['T0', 'T1', 'T2', 'T3']

# Build descriptive stats
desc_stats = []
for m in moments:
    vals = rt_subj[f'{m}_mean'].dropna()
    desc_stats.append({
        'Moment': m,
        'Mean': vals.mean(),
        'Median': vals.median(),
        'Min': vals.min(),
        'Max': vals.max()
    })
desc_df = pd.DataFrame(desc_stats)

# ---- Figure 1: Mean with range --------------------------------------------------
plt.figure(figsize=(6, 4))
for idx, row in desc_df.iterrows():
    plt.vlines(row['Moment'], row['Min'], row['Max'])
    plt.scatter(row['Moment'], row['Mean'], marker='o', s=60, label='Mean' if idx == 0 else None)

plt.title('Reaction Time – Mean with subject range')
plt.ylabel('RT (ms)')
plt.tight_layout()
plt.show()

# ---- Figure 2: Median with range ------------------------------------------------
plt.figure(figsize=(6, 4))
for idx, row in desc_df.iterrows():
    plt.vlines(row['Moment'], row['Min'], row['Max'])
    plt.scatter(row['Moment'], row['Median'], marker='s', s=60, label='Median' if idx == 0 else None)

plt.title('Reaction Time – Median with subject range')
plt.ylabel('RT (ms)')
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
#   RT analysis – MEDIAN of the three iterations (T0 … T3)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.stats.anova import AnovaRM
from scipy.stats import friedmanchisquare

# -------- build subject × moment table ----------------------
rt_rows_m = []
for subj, d in rtByPatient.items():
    rt_rows_m.append({'Subject': subj,
                      'RT_T0': np.nanmedian(d['T0']) if d['T0'] else np.nan,
                      'RT_T1': np.nanmedian(d['T1']) if d['T1'] else np.nan,
                      'RT_T2': np.nanmedian(d['T2']) if d['T2'] else np.nan,
                      'RT_T3': np.nanmedian(d['T3']) if d['T3'] else np.nan})
rt_subj_m = pd.DataFrame(rt_rows_m)

# -------- ANOVA (complete cases) ----------------------------
complete_m = rt_subj_m.dropna()
rt_long_m  = complete_m.melt(id_vars='Subject', var_name='Time', value_name='RT')
rt_long_m['Time'] = rt_long_m['Time'].str[-2:]

anova_m = AnovaRM(rt_long_m, 'RT', 'Subject', within=['Time']).fit()
print(anova_m)

# -------- Friedman on all available -------------------------
fr_m = rt_subj_m[['RT_T0','RT_T1','RT_T2','RT_T3']].dropna()
print('Friedman:', friedmanchisquare(*[fr_m[c] for c in fr_m]))

# -------- Plot ---------------------------------------------
desc_m = rt_long_m.groupby('Time')['RT'].agg(['median','sem'])
plt.errorbar(desc_m.index, desc_m['median'], yerr=desc_m['sem'],
             marker='o', capsize=5)
plt.ylabel('RT (ms)'); plt.title('Reaction time – MEDIAN')
plt.show()


In [ ]:
from scipy.stats import shapiro

# Supposons que votre DataFrame est nommé df et contient déjà les données

# Obtenir toutes les combinaisons uniques des facteurs
combinations = df[['Song', 'Condition', 'Feedback']].drop_duplicates()

# Initialiser une liste pour sauvegarder les résultats
results = []

for _, row in combinations.iterrows():
    # Filtrer df pour la combinaison actuelle de facteurs
    subset = df[(df['Song'] == row['Song']) & 
                (df['Condition'] == row['Condition']) & 
                (df['Feedback'] == row['Feedback'])]
    
    # Appliquer le test de Shapiro-Wilk aux scores de ce sous-ensemble
    stat, p = shapiro(subset['Score'])
    
    # Ajouter les résultats à la liste
    results.append({'Song': row['Song'], 'Condition': row['Condition'], 'Feedback': row['Feedback'], 'Shapiro-Wilk Stat': stat, 'p-value': p})

# Convertir les résultats en DataFrame pour une meilleure lisibilité
results_df = pd.DataFrame(results)

# Afficher les résultats
print(results_df)


In [ ]:
# Modèle ANOVA avec interaction entre les facteurs
model = ols('Score ~ C(Song) * C(Condition) * C(Feedback)', data=df).fit()

# Réalisation de l'ANOVA
anova_results = sm.stats.anova_lm(model, typ=2) # ANOVA de type II

# Affichage des résultats
print(anova_results)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Vos données
# Supposons que df soit votre DataFrame

# Modèle ANOVA avec interaction entre les facteurs
model = ols('Score ~ C(Song) * C(Condition) * C(Feedback)', data=df).fit()

# Réalisation de l'ANOVA
anova_results = sm.stats.anova_lm(model, typ=2) # ANOVA de type II
anova_results = anova_results.round(2)  # Arrondir les valeurs à deux décimales

# Enlever la ligne 'Residual'
# anova_results = anova_results.drop('Residual')

# Remplacer les petites valeurs p par "p < 0.001"
anova_results['PR(>F)'] = anova_results['PR(>F)'].apply(lambda x: 'p < 0.001' if x < 0.001 else x)

# Convertir les résultats de l'ANOVA en DataFrame pour un affichage plus attrayant
anova_table = pd.DataFrame(anova_results)

# Affichage des résultats sous forme de tableau
fig, ax = plt.subplots(figsize=(12, 2 + 0.5*len(anova_table)))  # Ajustez la taille selon le nombre de lignes
ax.axis('tight')
ax.axis('off')
the_table = ax.table(cellText=anova_table.values, colLabels=anova_table.columns, rowLabels=anova_table.index, cellLoc = 'center', loc='center', colWidths=[0.2]*len(anova_table.columns))
the_table.auto_set_font_size(False)
the_table.set_fontsize(12)
the_table.scale(1.5, 1.5)  # Ajuster l'échelle pour agrandir le tableau

# Ajouter un titre
plt.title('ANOVA Results for Interaction Between Rhythm, Condition, and Feedback')

plt.savefig('anova_table.png')  # Sauvegardez l'image pour l'insérer dans PowerPoint
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 1.  Add binary Melody / Speech factors derived from 'Condition'
# ------------------------------------------------------------------
# Condition codes in your data:  B, M, S, SM
df = df.copy()                                                # keep original intact
df["Melody"] = np.where(df["Condition"].isin(["M", "SM"]), "Yes", "No")
df["Speech"] = np.where(df["Condition"].isin(["S", "SM"]), "Yes", "No")

# ------------------------------------------------------------------
# 2.  Run the ANOVA with Melody × Speech (+ Song, Feedback)
# ------------------------------------------------------------------
from statsmodels.formula.api import ols
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas as pd

model_ms = ols(
    "Score ~ C(Song) * C(Melody) * C(Speech) * C(Feedback)",
    data=df
).fit()

anova_ms = sm.stats.anova_lm(model_ms, typ=2)       # same type‑II test
anova_ms = anova_ms.round(2)

# Optional: keep the same p‑value formatting as in your notebook
anova_ms["PR(>F)"] = anova_ms["PR(>F)"].apply(
    lambda p: "p < 0.001" if p < 0.001 else p
)

# ------------------------------------------------------------------
# 3.  Pretty‑print as a figure (matches your style)
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 2 + 0.5 * len(anova_ms)))
ax.axis("tight")
ax.axis("off")

table = ax.table(
    cellText=anova_ms.values,
    colLabels=anova_ms.columns,
    rowLabels=anova_ms.index,
    cellLoc="center",
    loc="center",
    colWidths=[0.2] * len(anova_ms.columns)
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.5, 1.5)

plt.title("ANOVA Results for Melody × Speech × Feedback × Song")
plt.savefig("anova_table_melody_speech.png")  # stays consistent with your pipeline
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene, ttest_ind, f_oneway, normaltest, mannwhitneyu, kruskal
from numpy import percentile

sns.set()

def remove_outliers(df, group_def):
    for song in group_def['songs']:
        for condition in group_def['conditions']:
            for feedback in ([group_def['feedback']] if group_def['feedback'] != 'Both' else ['Yes', 'No']):
                # Filtre selon les critères spécifiques
                criteria = (df['Song'] == song) & (df['Condition'] == condition) & (df['Feedback'] == feedback)
                sub_df = df[criteria]
                scores = sub_df['Score']
                q25, q75 = percentile(scores, 25), percentile(scores, 75)
                iqr = q75 - q25
                cut_off = iqr * 1.5
                lower, upper = q25 - cut_off, q75 + cut_off
                median = scores.median()
                # Remplacement des outliers par la médiane
                df.loc[criteria, 'Score'] = scores.apply(lambda x: median if x < lower or x > upper else x)
    return df

def analyze_groups(df, group_definitions, outliers=False):
    if outliers:  # Si le traitement des outliers est demandé
        for group_def in group_definitions:
            df = remove_outliers(df, group_def)
    
    means_list, group_normality = [], []
    for group_def in group_definitions:
        conditions = (df['Song'].isin(group_def['songs'])) & (df['Condition'].isin(group_def['conditions']))
        if group_def['feedback'] != 'Both':
            conditions &= (df['Feedback'] == group_def['feedback'])
        means = df[conditions].groupby('Subject')['Score'].mean()
        means_list.append(means)
        stat, p = normaltest(means.dropna())
        group_normality.append((group_def['name'], p, 'Normal' if p > 0.05 else 'Not normal'))
    
    # Boxplot
    combined_means = pd.concat(means_list, axis=1)
    combined_means.columns = [g['name'] for g in group_definitions]
    plt.figure(figsize=(10, 7))
    combined_means.boxplot(widths=0.8, showmeans=True, meanline=True)
    plt.title(' vs '.join([g['name'] for g in group_definitions]) + ' - Mean Errors Comparison')
    plt.ylabel('Error (%)')
    plt.legend([plt.Line2D([0], [0], color='green', linestyle='--', linewidth=1, label='Mean'),
                plt.Line2D([0], [0], color='grey', linestyle='-', linewidth=1, label='Median')],
               ['Mean', 'Median'])
    plt.show()
    
    # Scatter plot for two groups
    if len(means_list) == 2:
        plt.figure(figsize=(8, 8))
        sns.scatterplot(x=means_list[0], y=means_list[1])
        plt.title('Distribution Comparison - ' + ' vs '.join([g['name'] for g in group_definitions]))
        plt.xlabel(f'Error (%) - {group_definitions[0]["name"]}')
        plt.ylabel(f'Error (%) - {group_definitions[1]["name"]}')
        min_limit = min(means_list[0].min(), means_list[1].min()) * 0.95
        max_limit = max(means_list[0].max(), means_list[1].max()) * 1.05
        plt.xlim(min_limit, max_limit)
        plt.ylim(min_limit, max_limit)
        plt.plot([min_limit, max_limit], [min_limit, max_limit], 'r--')
        plt.plot(plt.xlim(), plt.ylim(), ls="--", c=".3")
        plt.gca().set_aspect('equal', adjustable='box')
        plt.show()

    # Display normality status with p-values
    for name, p, norm in group_normality:
        print(f"Normality test p-value: {p:.3f} ; {name} group is {norm}")

    # Statistical tests
    if all(norm == 'Normal' for _, _, norm in group_normality):
        if len(means_list) == 2:
            stat, p = ttest_ind(means_list[0].dropna(), means_list[1].dropna(), equal_var=False)
            print(f"t-test: p={p:.3e}")
        else:
            f_stat, p_value = f_oneway(*[x.dropna() for x in means_list])
            print(f"ANOVA: p-value={p_value:.3e}")
    else:  # Utiliser des tests non paramétriques si les données ne sont pas normales
        if len(means_list) == 2:
            stat, p = mannwhitneyu(means_list[0].dropna(), means_list[1].dropna())
            print(f"Mann-Whitney U test: p={p:.3e}")
        else:
            stat, p = kruskal(*means_list)
            print(f"Kruskal-Wallis test: p={p:.3e}")



In [ ]:
group_definitions_SAVED = [
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['S', 'SM'],
        'feedback': 'Both',
        'name': 'Paroles'
    },
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['B', 'M'],
        'feedback': 'Both',  
        'name': 'Notes'
    },
    {
        'songs': ['Balou'],
        'conditions': ['B', 'M', 'S', 'SM'],
        'feedback': 'Both',
        'name': 'Balou'
    },
    {
        'songs': ['Papaoutai'],
        'conditions': ['B', 'M', 'S', 'SM'],
        'feedback': 'Both',
        'name': 'Papaoutai'
    },
    {
        'songs': ['Piaf'],
        'conditions': ['B', 'M', 'S', 'SM'],
        'feedback': 'Both',  
        'name': 'Piaf'
    },
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['B'],
        'feedback': 'Both',
        'name': 'Single Note'
    },
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['M'],
        'feedback': 'Both',
        'name': 'Melody'
    },
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['S'],
        'feedback': 'Both',
        'name': 'Speech'
    },
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['SM'],
        'feedback': 'Both',
        'name': 'Singing'
    },
]

group_definitions = [

    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['M', 'SM'],
        'feedback': 'Both',
        'name': 'Melodic'
    },
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['B', 'S'],
        'feedback': 'Both',  
        'name': 'Not Melodic'
    },
]

analyze_groups(df, group_definitions, outliers=True)


TEST






-------------------------

In [ ]:
# Compter le nombre de musiciens (en assumant que la colonne s'appelle 'Musician')
number_of_musicians = df['Musician'].apply(lambda x: x[0] == 'oui').sum()/24

print(f"Nombre de musiciens dans la DataFrame : {number_of_musicians}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene, ttest_ind, f_oneway, normaltest, mannwhitneyu, kruskal
from numpy import percentile

sns.set()

def remove_outliers(df, group_def):
    for song in group_def['songs']:
        for condition in group_def['conditions']:
            for feedback in ([group_def['feedback']] if group_def['feedback'] != 'Both' else ['Yes', 'No']):
                # Filtre selon les critères spécifiques
                criteria = (df['Song'] == song) & (df['Condition'] == condition) & (df['Feedback'] == feedback)
                sub_df = df[criteria]
                scores = sub_df['Score']
                q25, q75 = percentile(scores, 25), percentile(scores, 75)
                iqr = q75 - q25
                cut_off = iqr * 1.5
                lower, upper = q25 - cut_off, q75 + cut_off
                median = scores.median()
                # Remplacement des outliers par la médiane
                df.loc[criteria, 'Score'] = scores.apply(lambda x: median if x < lower or x > upper else x)
    return df

def analyze_groups(df, group_definitions, outliers=False):
    if outliers:  # Si le traitement des outliers est demandé
        for group_def in group_definitions:
            df = remove_outliers(df, group_def)
    
    means_list, group_normality = [], []
    for group_def in group_definitions:
        conditions = (df['Song'].isin(group_def['songs'])) & (df['Condition'].isin(group_def['conditions']))
        
        # Filtrage supplémentaire pour feedback
        if group_def['feedback'] != 'Both':
            conditions &= (df['Feedback'] == group_def['feedback'])
        
        # Modification de la condition pour éviter l'erreur

        # Filtrage pour les musiciens
        if group_def['Musician'] == 'oui':
            conditions &= (df['Musician'] == 'oui') & (df['Musician_years'] >= group_def['Years'])
        elif group_def['Musician'] == 'non':
            conditions &= (df['Musician'] == 'non')
        # Pas de filtrage supplémentaire nécessaire pour "Both"
        
        # Filtrage pour les joueurs
        if group_def['Gamer'] == 'oui':
            conditions &= (df['Gamer'] == 'oui')
        elif group_def['Gamer'] == 'non':
            conditions &= (df['Gamer'] == 'non')
        # Pas de filtrage supplémentaire nécessaire pour "Both"
        
        means = df[conditions].groupby('Subject')['Score'].mean()
        means_list.append(means)
        # Vérification pour s'assurer qu'il y a suffisamment de données pour le test de normalité
        if len(means.dropna()) >= 8:
            stat, p = normaltest(means.dropna())
            group_normality.append((group_def['name'], p, 'Normal' if p > 0.05 else 'Not normal'))
        else:
            group_normality.append((group_def['name'], None, 'Insufficient data for normality test'))
    
    
    # Boxplot
    combined_means = pd.concat(means_list, axis=1)
    combined_means.columns = [g['name'] for g in group_definitions]
    plt.figure(figsize=(10, 7))
    combined_means.boxplot(widths=0.8, showmeans=True, backgroundcolor = "white", meanline=True)
    plt.title(' vs '.join([g['name'] for g in group_definitions]) + ' - Mean Errors Comparison')
    plt.ylabel('Error (%)')
    plt.legend([plt.Line2D([0], [0], color='green', linestyle='--', linewidth=1, label='Mean'),
                plt.Line2D([0], [0], color='grey', linestyle='-', linewidth=1, label='Median')],
               ['Mean', 'Median'])
    plt.show()
    
    # Scatter plot for two groups
    if len(means_list) == 2:
        plt.figure(figsize=(8, 8))
        sns.scatterplot(x=means_list[0], y=means_list[1])
        plt.title('Distribution Comparison - ' + ' vs '.join([g['name'] for g in group_definitions]))
        plt.xlabel(f'Error (%) - {group_definitions[0]["name"]}')
        plt.ylabel(f'Error (%) - {group_definitions[1]["name"]}')
        min_limit = min(means_list[0].min(), means_list[1].min()) * 0.95
        max_limit = max(means_list[0].max(), means_list[1].max()) * 1.05
        plt.xlim(min_limit, max_limit)
        plt.ylim(min_limit, max_limit)
        plt.plot([min_limit, max_limit], [min_limit, max_limit], 'r--')
        plt.plot(plt.xlim(), plt.ylim(), ls="--", c=".3")
        plt.gca().set_aspect('equal', adjustable='box')
        plt.show()

    # Display normality status with p-values
    for name, p, norm in group_normality:
        if p is not None:
            print(f"Normality test p-value: {p:.3f} ; {name} group is {norm}")
        else:
            # Si p est None, évitez d'utiliser le formatage .3f
            print(f"{name} group has insufficient data for normality test")


    # Statistical tests
    if all(norm == 'Normal' for _, _, norm in group_normality):
        if len(means_list) == 2:
            stat, p = ttest_ind(means_list[0].dropna(), means_list[1].dropna(), equal_var=False)
            print(f"t-test: p={p:.3e}")
        else:
            f_stat, p_value = f_oneway(*[x.dropna() for x in means_list])
            print(f"ANOVA: p-value={p_value:.3e}")
    else:  # Utiliser des tests non paramétriques si les données ne sont pas normales
        if len(means_list) == 2:
            stat, p = mannwhitneyu(means_list[0].dropna(), means_list[1].dropna())
            print(f"Mann-Whitney U test: p={p:.3e}")
        else:
            stat, p = kruskal(*means_list)
            print(f"Kruskal-Wallis test: p={p:.3e}")



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene, ttest_ind, f_oneway, normaltest, mannwhitneyu, kruskal
from numpy import percentile

sns.set()
sns.set(style="white")

def remove_outliers(df, group_def):
    for song in group_def['songs']:
        for condition in group_def['conditions']:
            for feedback in ([group_def['feedback']] if group_def['feedback'] != 'Both' else ['Yes', 'No']):
                criteria = (df['Song'] == song) & (df['Condition'] == condition) & (df['Feedback'] == feedback)
                sub_df = df[criteria]
                scores = sub_df['Score']
                q25, q75 = percentile(scores, 25), percentile(scores, 75)
                iqr = q75 - q25
                cut_off = iqr * 1.5
                lower, upper = q25 - cut_off, q75 + cut_off
                median = scores.median()
                df.loc[criteria, 'Score'] = scores.apply(lambda x: median if x < lower or x > upper else x)
    return df

def analyze_groups(df, group_definitions, outliers=False):
    if outliers:
        for group_def in group_definitions:
            df = remove_outliers(df, group_def)
    
    means_list, group_normality = [], []
    for group_def in group_definitions:
        conditions = (df['Song'].isin(group_def['songs'])) & (df['Condition'].isin(group_def['conditions']))
        if group_def['feedback'] != 'Both':
            conditions &= (df['Feedback'] == group_def['feedback'])
        if group_def['Musician'] == 'oui':
            conditions &= (df['Musician'] == 'oui') & (df['Musician_years'] >= group_def['Years'])
        elif group_def['Musician'] == 'non':
            conditions &= (df['Musician'] == 'non')
        if group_def['Gamer'] == 'oui':
            conditions &= (df['Gamer'] == 'oui')
        elif group_def['Gamer'] == 'non':
            conditions &= (df['Gamer'] == 'non')
        
        means = df[conditions].groupby('Subject')['Score'].mean()
        means_list.append(means)
        if len(means.dropna()) >= 8:
            stat, p = normaltest(means.dropna())
            group_normality.append((group_def['name'], p, 'Normal' if p > 0.05 else 'Not normal'))
        else:
            group_normality.append((group_def['name'], None, 'Insufficient data for normality test'))
    
    # Boxplot with scatter points
    combined_means = pd.concat(means_list, axis=1)
    combined_means.columns = [g['name'] for g in group_definitions]
    plt.figure(figsize=(10, 7), facecolor='white')
    plt.gca().set_facecolor('white')
    plt.grid(False)
    ax = plt.gca()
    ax.xaxis.grid(False)  # Désactive la grille pour l'axe x
    ax.yaxis.grid(False)  # Désactive la grille pour l'axe y

    boxprops = dict(linestyle='-', linewidth=2, color='black', alpha=0.5)
    ax = combined_means.boxplot(widths=0.8, showmeans=True, meanline=True, patch_artist=True, return_type='dict', boxprops=boxprops)


    
    for patch in ax['boxes']:
        patch.set(facecolor='white')
    for i, mean in enumerate(means_list):
        plt.scatter(x=[i + 1] * len(mean), y=mean, color='red', alpha=1, s=1)
    # Désactiver la grille après la création du boxplot
    plt.gca().xaxis.grid(False)  # Désactive la grille pour l'axe des x
    plt.gca().yaxis.grid(False)  # Désactive la grille pour l'axe des y

    for patch in ax['boxes']:
        patch.set(facecolor='white')
    for i, mean in enumerate(means_list):
        plt.scatter(x=[i + 1] * len(mean), y=mean, color='red', alpha=1, s=5)

    # Ajouter la légende pour la médiane et la moyenne
    median_patch = plt.Line2D([], [], color='grey', linestyle='-', linewidth=1, label='Median')
    mean_patch = plt.Line2D([], [], color='green', linestyle='--', linewidth=1, label='Mean')
    plt.legend(handles=[mean_patch, median_patch], loc='upper left')
    plt.title(' vs '.join([g['name'] for g in group_definitions]) + ' - Mean Errors Comparison')
    plt.ylabel('Error (%)')
    plt.show()

    # Display normality status with p-values
    for name, p, norm in group_normality:
        if p is not None:
            print(f"Normality test p-value: {p:.3f} ; {name} group is {norm}")
        else:
            print(f"{name} group has insufficient data for normality test")

    # Statistical tests
    if all(norm == 'Normal' for _, _, norm in group_normality):
        if len(means_list) == 2:
            stat, p = ttest_ind(means_list[0].dropna(), means_list[1].dropna(), equal_var=False)
            print(f"t-test: p={p:.3e}")
        else:
            f_stat, p_value = f_oneway(*[x.dropna() for x in means_list])
            print(f"ANOVA: p-value={p_value:.3e}")
    else:
        if len(means_list) == 2:
            stat, p = mannwhitneyu(means_list[0].dropna(), means_list[1].dropna())
            print(f"Mann-Whitney U test: p={p:.3e}")
        else:
            stat, p = kruskal(*means_list)
            print(f"Kruskal-Wallis test: p={p:.3e}")


In [ ]:
def combined_song_condition(df):
    """
    Draw one figure with two panels:
      A) box-and-scatter (mean error per subject) for each Song
      B) box-and-scatter (mean error per subject) for each Condition
    Styling matches the existing analyze_groups() design.
    """
    # ------------------------------------------------------------
    # 1) compute per-subject means
    # ------------------------------------------------------------
    song_pivot = (
        df.groupby(['Subject', 'Song'])['Score']
          .mean()
          .unstack('Song')
          .sort_index(axis=1)          # stable column order
    )

    cond_pivot = (
        df.groupby(['Subject', 'Condition'])['Score']
          .mean()
          .unstack('Condition')
          .sort_index(axis=1)
    )

    # ------------------------------------------------------------
    # 2) prepare figure & common style
    # ------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True,
                             facecolor='white')
    for ax in axes:
        ax.set_facecolor('white')
        ax.xaxis.grid(False)
        ax.yaxis.grid(False)

    boxprops = dict(linestyle='-', linewidth=2, color='black', alpha=0.5)

    # helper to draw one panel -----------------------------------
    def _draw_boxscatter(pivot_df, ax, title):
        b = pivot_df.boxplot(ax=ax, widths=0.8,
                             showmeans=True, meanline=True,
                             patch_artist=True, boxprops=boxprops,
                             return_type='dict')
        # white fill
        for patch in b['boxes']:
            patch.set(facecolor='white')

        # red dots (subject means)
        for i, col in enumerate(pivot_df.columns):
            ax.scatter(np.full(pivot_df[col].shape, i+1),
                       pivot_df[col], color='red', s=6, alpha=1)

        # legend for mean / median
        median_line = plt.Line2D([], [], color='grey', linestyle='-',
                                 linewidth=1, label='Median')
        mean_line   = plt.Line2D([], [], color='green', linestyle='--',
                                 linewidth=1, label='Mean')
        ax.legend(handles=[mean_line, median_line], loc='upper left')
        ax.set_title(title)
        ax.set_ylabel('Tapping error (%)')
        ax.xaxis.grid(False)
        ax.yaxis.grid(False)

    # ------------------------------------------------------------
    # 3) draw panels
    # ------------------------------------------------------------
    _draw_boxscatter(song_pivot, axes[0], 'A) Error distribution by Song')
    # ---- B) Condition -------------------------------------------------------
    # ❶ OPTIONAL: rename condition codes to more explicit short labels
    label_map = {
        'B' : 'Mon-Note',      # monotonous note
        'M' : 'Mel-Note',      # melodic note
        'S' : 'Mon-Speech',    # monotonous speech
        'SM': 'Mel-Speech'     # melodic speech
    }
    cond_pivot.rename(columns=label_map, inplace=True)

    # ❷ now draw the panel with those new labels
    _draw_boxscatter(cond_pivot, axes[1], 'B) Error distribution by Condition')

    plt.tight_layout()
    plt.show()
# assume df is already loaded + cleaned
combined_song_condition(df)
# optionally save:
# plt.savefig('Fig_Song_Condition_Boxplot.png', dpi=300)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

###############################################################################
# Melody × Semantic figure with multiple significance bars                    #
###############################################################################

def draw_melody_semantic_boxplots(
    df: pd.DataFrame,
    *,
    fontsize: int = 9,
    # p‑values for the simple/main effects
    p_melody: float = 1e-3,
    p_semantic: float = 1e-3,
    # pairwise comparisons within the interaction panel
    p_note_melody: float = 1e-3,      # Mon‑Note vs Mel‑Note
    p_monnote_monspeech: float = 1e-3,  # Mon‑Note vs Mon‑Speech
    p_monnote_melspeech: float = 1e-3,  # Mon‑Note vs Mel‑Speech
    p_speech_melody: float = 0.96     # Mon‑Speech vs Mel‑Speech
) -> None:
    """Draw Melody/Semantic box‑plots **with extended significance bars**.

    Additional pairwise bars in the interaction panel:
      • Mon‑Note vs Mon‑Speech (`p_monnote_monspeech`)
      • Mon‑Note vs Mel‑Speech (`p_monnote_melspeech`)

    All p‑value arguments convert to stars via common thresholds.
    """

    # ------------------------------------------------------------------
    # Factor decoding ---------------------------------------------------
    # ------------------------------------------------------------------
    melody_map   = {'B': 'Monotone', 'M': 'Melodic', 'S': 'Monotone', 'SM': 'Melodic'}
    semantic_map = {'B': 'Note',     'M': 'Note',    'S': 'Speech',   'SM': 'Speech'}
    combo_map    = {'B': 'Mon-Note', 'M': 'Mel-Note', 'S': 'Mon-Speech', 'SM': 'Mel-Speech'}

    df = df.copy()
    df['Melody']   = df['Condition'].map(melody_map)
    df['Semantic'] = df['Condition'].map(semantic_map)
    df['Combo']    = df['Condition'].map(combo_map)

    # Helper ------------------------------------------------------------
    def _pivot(s_means: pd.DataFrame, col: str) -> pd.DataFrame:
        return s_means.pivot(index='Subject', columns=col, values='Score').sort_index(axis=1)

    def _draw_boxscatter(pivot_df: pd.DataFrame, ax: plt.Axes, title: str,
                         ylab: str = 'Tapping error (%)', legend: bool = False):
        b = pivot_df.boxplot(ax=ax, widths=0.8, showmeans=True, meanline=True,
                             patch_artist=True,
                             boxprops=dict(linestyle='-', linewidth=2, color='black', alpha=0.6),
                             return_type='dict')
        for p in b['boxes']:
            p.set(facecolor='white')
        for i, col in enumerate(pivot_df.columns):
            ax.scatter(np.full(pivot_df[col].shape, i+1), pivot_df[col],
                       color='red', s=8, alpha=0.9)
        if legend:
            ax.legend([
                plt.Line2D([], [], color='green', linestyle='--', linewidth=1, label='Mean'),
                plt.Line2D([], [], color='grey',  linestyle='-',  linewidth=1, label='Median')
            ], ['Mean', 'Median'], loc='upper left', frameon=False)
        ax.set_title(title)
        ax.set_ylabel(ylab)
        ax.xaxis.grid(False); ax.yaxis.grid(False)

    def _star(p):
        return '' if p >= 0.05 else ('*' if p >= 0.01 else ('**' if p >= 0.001 else '***'))

    # ------------------------------------------------------------------
    # Per‑subject means -------------------------------------------------
    # ------------------------------------------------------------------
    subj_cond = df.groupby(['Subject', 'Condition'])['Score'].mean().reset_index()
    subj_mel  = df.groupby(['Subject', 'Melody'])['Score'].mean().reset_index()
    subj_sem  = df.groupby(['Subject', 'Semantic'])['Score'].mean().reset_index()

    # ------------------------------------------------------------------
    # Plotting ----------------------------------------------------------
    # ------------------------------------------------------------------
    rc = {
        'font.size': fontsize,
        'axes.titlesize': fontsize + 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize - 1,
        'ytick.labelsize': fontsize - 1,
        'legend.fontsize': fontsize - 1,
    }
    with plt.rc_context(rc):
        fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True, facecolor='white')
        for ax in axes:
            ax.set_facecolor('white')

        # --- A) Melody main effect ------------------------------------
        mel_pivot = _pivot(subj_mel, 'Melody')
        _draw_boxscatter(mel_pivot, axes[0], 'A) Melody main effect', legend=True)
        yA = mel_pivot.max().max(); offA = yA*0.07
        axes[0].plot([1,1,2,2],[yA+offA,yA+1.15*offA,yA+1.15*offA,yA+offA], lw=1.2, color='black')
        axes[0].text(1.5, yA+1.22*offA, _star(p_melody), ha='center', va='bottom')

        # --- B) Semantic main effect ----------------------------------
        sem_pivot = _pivot(subj_sem, 'Semantic')
        _draw_boxscatter(sem_pivot, axes[1], 'B) Semantic main effect', '')
        yB = sem_pivot.max().max(); offB = yB*0.07
        axes[1].plot([1,1,2,2],[yB+offB,yB+1.15*offB,yB+1.15*offB,yB+offB], lw=1.2, color='black')
        axes[1].text(1.5, yB+1.22*offB, _star(p_semantic), ha='center')

        # --- C) Interaction -------------------------------------------
        subj_int = subj_cond.copy(); subj_int['Combo'] = subj_int['Condition'].map(combo_map)
        order = ['Mon-Note','Mel-Note','Mon-Speech','Mel-Speech']
        int_pivot = _pivot(subj_int, 'Combo').reindex(columns=order)
        _draw_boxscatter(int_pivot, axes[2], 'C) Melody × Semantic interaction', '')

        yC = int_pivot.max().max(); offC = yC*0.07
        # 1) Mon‑Note vs Mel‑Note (boxes 1‑2)
        axes[2].plot([1,1,2,2],[yC+offC,yC+1.15*offC,yC+1.15*offC,yC+offC], lw=1.2, color='black')
        axes[2].text(1.5, yC+1.22*offC, _star(p_note_melody), ha='center')
        # 2) Mon‑Note vs Mon‑Speech (boxes 1‑3)
        axes[2].plot([1,1,3,3],[yC+1.35*offC,yC+1.55*offC,yC+1.55*offC,yC+1.35*offC], lw=1.2, color='black')
        axes[2].text(2, yC+1.58*offC, _star(p_monnote_monspeech), ha='center')
        # 3) Mon‑Note vs Mel‑Speech (boxes 1‑4)
        axes[2].plot([1,1,4,4],[yC+1.8*offC,yC+2*offC,yC+2*offC,yC+1.8*offC], lw=1.2, color='black')
        axes[2].text(2.5, yC+2.03*offC, _star(p_monnote_melspeech), ha='center')
        # 4) Mon‑Speech vs Mel‑Speech (boxes 3‑4)
        #axes[2].plot([3,3,4,4],[yC+offC,yC+1.15*offC,yC+1.15*offC,yC+offC], lw=1.2, color='black')
        #axes[2].text(3.5, yC+1.22*offC, _star(p_speech_melody), ha='center')

        plt.tight_layout(); plt.show()

###############################################################################
# Example usage:
draw_melody_semantic_boxplots(df, fontsize=14,
                              p_melody=1e-4, p_semantic=1e-4,
                              p_note_melody=1e-4,
                              p_monnote_monspeech=1e-4,
                              p_monnote_melspeech=1e-4)
###############################################################################


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def draw_melody_semantic_boxplots(df, fontsize: int = 9):
    """Draw **box‑and‑scatter** panels (Melody main‑effect, Semantic main‑effect,
    Melody × Semantic interaction) as before, but with a user‑controllable
    *fontsize* argument.  Simply call, e.g. ::

        draw_melody_semantic_boxplots(df, fontsize=8)

    to shrink, or ``fontsize=14`` to enlarge all text elements in the figure.
    """

    # ------------------------------------------------------------------
    # 1) Factor coding --------------------------------------------------
    # ------------------------------------------------------------------
    melody_map   = {'B': 'Monotone', 'M': 'Melodic', 'S': 'Monotone', 'SM': 'Melodic'}
    semantic_map = {'B': 'Note',     'M': 'Note',    'S': 'Speech',   'SM': 'Speech'}
    combo_map    = {
        'B':  'Mon-Note',
        'M':  'Mel-Note',
        'S':  'Mon-Speech',
        'SM': 'Mel-Speech'
    }

    df = df.copy()
    df['Melody']   = df['Condition'].map(melody_map)
    df['Semantic'] = df['Condition'].map(semantic_map)
    df['Combo']    = df['Condition'].map(combo_map)

    # ---------------------------------------------------------------
    # 2) Helper functions -------------------------------------------
    # ---------------------------------------------------------------
    def _pivot(subject_means, col):
        return (
            subject_means.pivot(index='Subject', columns=col, values='Score')
                        .sort_index(axis=1)
        )

    def _draw_boxscatter(pivot_df, ax, title, yaxis_label='Tapping error (%)'):
        boxprops = dict(linestyle='-', linewidth=2, color='black', alpha=0.5)
        b = pivot_df.boxplot(ax=ax, widths=0.8,
                             showmeans=True, meanline=True,
                             patch_artist=True, boxprops=boxprops,
                             return_type='dict')
        for patch in b['boxes']:
            patch.set(facecolor='white')
        for i, col in enumerate(pivot_df.columns):
            ax.scatter(np.full(pivot_df[col].shape, i+1),
                       pivot_df[col], color='red', s=8, alpha=0.9)
        # legend (only first panel)
        if title.startswith('A'):
            median_line = plt.Line2D([], [], color='grey', linestyle='-',
                                     linewidth=1, label='Median')
            mean_line   = plt.Line2D([], [], color='green', linestyle='--',
                                     linewidth=1, label='Mean')
            ax.legend(handles=[mean_line, median_line], loc='upper left',
                      frameon=False)
        ax.set_title(title)
        ax.set_ylabel(yaxis_label)
        ax.xaxis.grid(False)
        ax.yaxis.grid(False)

    # ---------------------------------------------------------------
    # 3) Compute per‑subject means ---------------------------------
    # ---------------------------------------------------------------
    subj_cond = df.groupby(['Subject', 'Condition'])['Score'].mean().reset_index()
    subj_mel  = df.groupby(['Subject', 'Melody'])['Score'].mean().reset_index()
    subj_sem  = df.groupby(['Subject', 'Semantic'])['Score'].mean().reset_index()

    # Use a temporary rc_context so changes remain local -------------
    with plt.rc_context({
        'font.size': fontsize,
        'axes.titlesize': fontsize + 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize - 1,
        'ytick.labelsize': fontsize - 1,
        'legend.fontsize': fontsize - 1,
    }):
        # -----------------------------------------------------------
        # 4) Prepare figure ----------------------------------------
        # -----------------------------------------------------------
        fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True,
                                 facecolor='white')
        for ax in axes:
            ax.set_facecolor('white')

        # Melody effect
        pivot_mel = _pivot(subj_mel, 'Melody')
        _draw_boxscatter(pivot_mel, axes[0], 'A) Melody main effect')

        # Semantic effect
        pivot_sem = _pivot(subj_sem, 'Semantic')
        _draw_boxscatter(pivot_sem, axes[1], 'B) Semantic main effect', '')

        # Interaction
        subj_int = subj_cond.copy()
        subj_int['Combo'] = subj_int['Condition'].map(combo_map)
        order = ['Mon-Note', 'Mel-Note', 'Mon-Speech', 'Mel-Speech']
        pivot_int = (_pivot(subj_int, 'Combo').reindex(columns=order))
        _draw_boxscatter(pivot_int, axes[2], 'C) Melody × Semantic interaction', '')

        plt.tight_layout()
        plt.show()


# Example usage:
draw_melody_semantic_boxplots(df, fontsize=16)
# plt.savefig('Fig_Melody_Semantic_Boxplots.png', dpi=300)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Style par défaut, police un peu plus grande
plt.style.use('default')
plt.rcParams.update({'font.size': 22})

def plot_three_panel_by_song(df):
    """
    Figure 3×n par chanson :
      A) Erreur globale
      B) Monotone vs Melodic (Note only)
      C) Note vs Speech (Monotone only)
    """
    songs = sorted(df['Song'].unique())
    n = len(songs)

    # Prépa des données
    song_data = [df.loc[df['Song']==s, 'Score'].values for s in songs]
    df_note  = df[df['Speech_lab']=='Note']
    melody_data = [
        [df_note.loc[(df_note['Song']==s)&(df_note['Melody_lab']==lab), 'Score'].values
         for lab in ('Monotone','Melodic')]
        for s in songs
    ]
    df_mono = df[df['Melody_lab']=='Monotone']
    speech_data = [
        [df_mono.loc[(df_mono['Song']==s)&(df_mono['Speech_lab']==lab), 'Score'].values
         for lab in ('Note','Speech')]
        for s in songs
    ]

    # Création de la grille, hauteur réduite
    fig, axes = plt.subplots(3, n, figsize=(5*n, 12), sharey='row')
    fig.subplots_adjust(hspace=0.4, wspace=0.3)

    # Propriétés boxplot (contours actifs)
    boxprops    = dict(linestyle='-', linewidth=2, color='black', alpha=0.5)
    whiskerprops= dict(linestyle='-', linewidth=2, color='black', alpha=0.5)
    capprops    = dict(linewidth=2, color='black', alpha=0.5)
    meanprops   = dict(linestyle='--', linewidth=1, color='grey')
    medianprops = dict(linestyle='-',  linewidth=1, color='grey')

    ylabs = ['Tapping error (%)'] * 3

    for row in range(3):
        data_list = [song_data, melody_data, speech_data][row]

        for col, song in enumerate(songs):
            ax = axes[row, col]
            data = data_list[col]

            # on enlève les spines (cadre)
            for spine in ax.spines.values():
                spine.set_visible(False)

            # fond blanc sans grille
            ax.set_facecolor('white')
            ax.grid(False)

            # boxplot
            b = ax.boxplot(
                data, widths=0.6, patch_artist=True,
                boxprops=boxprops,
                whiskerprops=whiskerprops,
                capprops=capprops,
                meanprops=meanprops,
                medianprops=medianprops,
                showmeans=True, meanline=True
            )
            for patch in b['boxes']:
                patch.set(facecolor='white')

            # points individuels
            if row == 0:
                ax.scatter(np.ones_like(data), data, color='red', s=8)
            else:
                for i, subset in enumerate(data):
                    ax.scatter(np.full(len(subset), i+1), subset, color='red', s=8)

            # titre colonne (seulement 1ʳᵉ rangée)
            if row == 0:
                ax.set_title(song, fontsize=24)

            # étiquettes x
            if row == 1:
                ax.set_xticks([1,2])
                ax.set_xticklabels(['Monotone','Melodic'], fontsize=20)
            elif row == 2:
                ax.set_xticks([1,2])
                ax.set_xticklabels(['Note','Speech'], fontsize=20)
            else:
                ax.set_xticks([])

            # y-label seulement sur la 1ʳᵉ colonne
            if col == 0:
                ax.set_ylabel(ylabs[row], fontsize=22)

    plt.tight_layout(rect=[0.05, 0, 1, 1])
    plt.show()

plot_three_panel_by_song(df_prep)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ───────────────────────────────────────────────────────────────
# 1. Fonctions de calcul
# ───────────────────────────────────────────────────────────────
def compute_relative_effects(df):
    """
    Calcule les différences absolues (points de %) et relatives (%)
    pour :
      • Melody vs Monotone (Note only),
      • Speech vs Note     (Monotone only),
    à la fois au global et par chanson.
    Retourne un dictionnaire 'effects'.
    """
    # Sous‑ensembles utiles
    df_note      = df[df['Speech_lab']  == 'Note']
    df_speech    = df[df['Speech_lab']  == 'Speech']
    df_monotone  = df[df['Melody_lab']  == 'Monotone']
    df_melodic   = df[df['Melody_lab']  == 'Melodic']

    # ── Effet Melody (Note uniquement) : global
    m_monotone = df_note[df_note['Melody_lab']=='Monotone']['Score'].mean()
    m_melodic  = df_note[df_note['Melody_lab']=='Melodic' ]['Score'].mean()
    melody_abs = m_monotone - m_melodic                       # pts de %
    melody_rel = melody_abs / m_monotone * 100                # %

    # ── Effet Speech (Monotone uniquement) : global
    s_note   = df_monotone[df_monotone['Speech_lab']=='Note'  ]['Score'].mean()
    s_speech = df_monotone[df_monotone['Speech_lab']=='Speech']['Score'].mean()
    speech_abs = s_note - s_speech
    speech_rel = speech_abs / s_note * 100

    # ── Effets par chanson
    songs = sorted(df['Song'].unique())
    melody_stats, speech_stats = [], []
    for song in songs:
        # Melody‑vs‑Monotone
        mono = df_note[   (df_note['Song']==song)&(df_note['Melody_lab']=='Monotone')]['Score'].mean()
        mel  = df_note[   (df_note['Song']==song)&(df_note['Melody_lab']=='Melodic' )]['Score'].mean()
        melody_stats.append(
            (song,
             mono - mel,                       # abs
            (mono - mel) / mono * 100)         # rel
        )

        # Speech‑vs‑Note
        note = df_monotone[(df_monotone['Song']==song)&(df_monotone['Speech_lab']=='Note'  )]['Score'].mean()
        sp   = df_monotone[(df_monotone['Song']==song)&(df_monotone['Speech_lab']=='Speech')]['Score'].mean()
        speech_stats.append(
            (song,
             note - sp,
            (note - sp) / note * 100)
        )

    return {
        'overall_melody_abs': melody_abs,
        'overall_melody_rel': melody_rel,
        'overall_speech_abs': speech_abs,
        'overall_speech_rel': speech_rel,
        'melody_stats': melody_stats,   # list of (song, abs, rel)
        'speech_stats': speech_stats
    }

# ───────────────────────────────────────────────────────────────
# 2. Fonction de tracé
# ───────────────────────────────────────────────────────────────
def plot_relative_effects(effects):
    """
    Bar‑plot : amélioration relative (%) par chanson.
    Une barre bleue = Melody vs Monotone
    Une barre orange = Speech vs Note
    """
    songs       = [t[0] for t in effects['melody_stats']]
    melody_rel  = [t[2] for t in effects['melody_stats']]
    speech_rel  = [t[2] for t in effects['speech_stats']]

    ind   = np.arange(len(songs))
    width = 0.35

    fig, ax = plt.subplots(figsize=(1 + 1.7*len(songs), 6))
    bars1   = ax.bar(ind,          melody_rel,  width, label='Melody vs Monotone')
    bars2   = ax.bar(ind + width,  speech_rel,  width, label='Speech vs Note')

    # Axe & titres
    ax.axhline(0, color='black', linewidth=1)
    ax.set_ylabel('Relative improvement (%)')
    ax.set_title('Relative reduction in tapping error per song')
    ax.set_xticks(ind + width / 2)
    ax.set_xticklabels(songs, rotation=45, ha='right')
    ax.legend(frameon=False)

    # Annotations au‑dessus de chaque barre
    for bars in (bars1, bars2):
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.0f}%',
                        xy=(bar.get_x() + bar.get_width()/2, height),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()

# ───────────────────────────────────────────────────────────────
# 3. Exemple d’utilisation (df_prep déjà défini plus haut)
# ───────────────────────────────────────────────────────────────
effects = compute_relative_effects(df_prep)

# (optionnel) aperçu texte des chiffres
print(f"Overall Melody effect:  {effects['overall_melody_abs']:.2f} pp  "
      f"({effects['overall_melody_rel']:.0f} %)")
print(f"Overall Speech effect:  {effects['overall_speech_abs']:.2f} pp  "
      f"({effects['overall_speech_rel']:.0f} %)")
for (song, a, r) in effects['melody_stats']:
    print(f"{song:10s}  Melody effect: {a:.2f} pp ({r:.0f} %)")
for (song, a, r) in effects['speech_stats']:
    print(f"{song:10s}  Speech  effect: {a:.2f} pp ({r:.0f} %)")

# tracé
plot_relative_effects(effects)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

###############################################################################
# Shared helper functions & style                                              #
###############################################################################

def _pivot(subject_means: pd.DataFrame, col: str) -> pd.DataFrame:
    """Return a Subjects × Factor‑Level pivot table sorted by columns."""
    return (
        subject_means.pivot(index='Subject', columns=col, values='Score')
                     .sort_index(axis=1)
    )


def _draw_boxscatter(pivot_df: pd.DataFrame, ax: plt.Axes, title: str,
                     add_legend: bool = False) -> None:
    """Draw a white‑filled boxplot with red subject dots and optional legend."""
    boxprops = dict(linestyle='-', linewidth=2, color='black', alpha=0.6)
    b = pivot_df.boxplot(ax=ax, widths=0.8, showmeans=True, meanline=True,
                         patch_artist=True, boxprops=boxprops,
                         return_type='dict')
    for patch in b['boxes']:
        patch.set(facecolor='white')

    for i, col in enumerate(pivot_df.columns):
        ax.scatter(np.full(pivot_df[col].shape, i + 1), pivot_df[col],
                   color='red', s=10, alpha=0.9)

    if add_legend:
        median_line = plt.Line2D([], [], color='grey', linestyle='-', linewidth=1,
                                 label='Median')
        mean_line   = plt.Line2D([], [], color='green', linestyle='--', linewidth=1,
                                 label='Mean')
        ax.legend(handles=[mean_line, median_line], loc='upper left', frameon=False)

    ax.set_title(title)
    ax.set_ylabel('Tapping error (%)')
    ax.xaxis.grid(False)
    ax.yaxis.grid(False)

###############################################################################
# Figure 1: Melody × Semantic (unchanged)                                      #
###############################################################################

def draw_melody_semantic_boxplots(df: pd.DataFrame, *, fontsize: int = 9) -> None:
    """Box‑and‑scatter plots for Melody main effect, Semantic main effect, and
    their interaction. Usage::

        draw_melody_semantic_boxplots(df, fontsize=8)
    """

    melody_map   = {'B': 'Monotone', 'M': 'Melodic', 'S': 'Monotone', 'SM': 'Melodic'}
    semantic_map = {'B': 'Note',     'M': 'Note',    'S': 'Speech',   'SM': 'Speech'}
    combo_map    = {'B': 'Note‑Mon', 'M': 'Note‑Mel', 'S': 'Speech‑Mon', 'SM': 'Speech‑Mel'}

    df_ = df.copy()
    df_['Melody']   = df_['Condition'].map(melody_map)
    df_['Semantic'] = df_['Condition'].map(semantic_map)
    df_['Combo']    = df_['Condition'].map(combo_map)

    subj_cond = df_.groupby(['Subject', 'Condition'])['Score'].mean().reset_index()
    subj_mel  = df_.groupby(['Subject', 'Melody'])['Score'].mean().reset_index()
    subj_sem  = df_.groupby(['Subject', 'Semantic'])['Score'].mean().reset_index()

    with plt.rc_context({
        'font.size': fontsize,
        'axes.titlesize': fontsize + 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize - 1,
        'ytick.labelsize': fontsize - 1,
        'legend.fontsize': fontsize - 1,
    }):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True, facecolor='white')
        for ax in axes:
            ax.set_facecolor('white')

        _draw_boxscatter(_pivot(subj_mel, 'Melody'),   axes[0], 'A) Melody main effect', add_legend=True)
        _draw_boxscatter(_pivot(subj_sem, 'Semantic'), axes[1], 'B) Semantic main effect')

        subj_int = subj_cond.copy()
        subj_int['Combo'] = subj_int['Condition'].map(combo_map)
        order = ['Note‑Mon', 'Note‑Mel', 'Speech‑Mon', 'Speech‑Mel']
        _draw_boxscatter(_pivot(subj_int, 'Combo').reindex(columns=order),
                         axes[2], 'C) Melody × Semantic interaction')

        plt.tight_layout()
        plt.show()

###############################################################################
# Figure 2: Feedback × Semantic (updated – removed Semantic panel)             #
###############################################################################

def draw_feedback_semantic_boxplots(df: pd.DataFrame, *, fontsize: int = 9,
                                    p_note: float = 0.03,
                                    p_speech: float = 0.001) -> None:
    """Box‑and‑scatter figure for the **Feedback × Semantic** interaction.

    Layout: **two** panels
      • A) Feedback main effect (No FB vs FB)
      • B) Feedback × Semantic interaction (Note‑NoFB, Note‑FB, Speech‑NoFB, Speech‑FB)

    Significance bars with stars are drawn above the Note and Speech comparisons
    in the interaction panel. Parameters `p_note` and `p_speech` control the
    displayed p‑values (default 0.03 and 0.001).
    """

    # ------------------------- Factor extraction -----------------------------
    df_ = df.copy()

    # Semantic factor -> 'Note' / 'Speech'
    if 'Semantic' not in df_.columns:
        if 'Condition' not in df_.columns:
            raise KeyError("Need column 'Semantic' or 'Condition' in df.")
        semantic_map = {'B': 'Note', 'M': 'Note', 'S': 'Speech', 'SM': 'Speech'}
        df_['Semantic'] = df_['Condition'].map(semantic_map)

    # Feedback factor -> 'NoFB' / 'FB'
    if 'Feedback' not in df_.columns:
        for alt in ('FeedbackType', 'AuditoryFB'):
            if alt in df_.columns:
                df_['Feedback'] = df_[alt]
                break
        else:
            raise KeyError("Need column 'Feedback' (or 'FeedbackType' / 'AuditoryFB') in df.")

    fb_map = {0: 'NoFB', 1: 'FB', False: 'NoFB', True: 'FB',
              'No': 'NoFB', 'Yes': 'FB', 'Without': 'NoFB', 'With': 'FB'}
    df_['Feedback'] = df_['Feedback'].map(lambda x: fb_map.get(x, str(x)))

    # Interaction combination label
    df_['Combo'] = df_['Semantic'] + '-' + df_['Feedback']

    # ------------------------- Per‑subject means ----------------------------
    subj_fb  = df_.groupby(['Subject', 'Feedback'])['Score'].mean().reset_index()
    subj_int = df_.groupby(['Subject', 'Combo'])   ['Score'].mean().reset_index()

    order = ['Note-NoFB', 'Note-FB', 'Speech-NoFB', 'Speech-FB']

    # ------------------------- Plotting -------------------------------------
    with plt.rc_context({
        'font.size': fontsize,
        'axes.titlesize': fontsize + 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize - 1,
        'ytick.labelsize': fontsize - 1,
        'legend.fontsize': fontsize - 1,
    }):
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, facecolor='white')
        for ax in axes:
            ax.set_facecolor('white')

        # A) Feedback main effect
        _draw_boxscatter(_pivot(subj_fb, 'Feedback'), axes[0], 'A) Feedback main effect', add_legend=True)

        # B) Interaction
        pivot_int = _pivot(subj_int, 'Combo').reindex(columns=order)
        _draw_boxscatter(pivot_int, axes[1], 'B) Feedback × Semantic interaction')

        # Significance bars
        ax_int = axes[1]
        y_max = pivot_int.max().max()
        offset = y_max * 0.05

        def _star(p):
            return 'n.s.' if p >= 0.05 else ('*' if p >= 0.01 else ('**' if p >= 0.001 else '***'))

        # Note comparison (columns 1 & 2)
        ax_int.plot([1, 1, 2, 2], [y_max + offset, y_max + offset*1.2, y_max + offset*1.2, y_max + offset],
                    lw=1.2, color='black')
        ax_int.text(1.5, y_max + offset*1.3, _star(p_note), ha='center', va='bottom')

        # Speech comparison (columns 3 & 4)
        ax_int.plot([3, 3, 4, 4], [y_max + offset, y_max + offset*1.2, y_max + offset*1.2, y_max + offset],
                    lw=1.2, color='black')
        ax_int.text(3.5, y_max + offset*1.3, _star(p_speech), ha='center', va='bottom')

        plt.tight_layout()
        plt.show()

###############################################################################
# End of module                                                               #
###############################################################################


# utilisation typique
draw_feedback_semantic_boxplots(df, fontsize=12, p_note=0.03, p_speech=0.001)
# utilisation typique



In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

###############################################################################
# 0) USER SETTINGS
###############################################################################
# If your column names differ, map them here.
COLUMN_ALIASES = {
    'subject'   : 'Subject',
    'song'      : 'Song',
    'condition' : 'Condition',   # categorical: 'B','M','S','SM' or strings
    'score'     : 'Score',       # numeric tapping error (%)
    'feedback'  : 'Feedback'     # 0/1 or strings like 'No','Yes'
}

# Mapping from your 4 legacy conditions to the 2×2 factors
# Edit if your codes differ.
CONDITION_TO_FACTORS = {
    'B' : {'Melody':0, 'Speech':0},   # Monotone Note
    'M' : {'Melody':1, 'Speech':0},   # Melodic  Note
    'S' : {'Melody':0, 'Speech':1},   # Monotone Speech
    'SM': {'Melody':1, 'Speech':1},   # Melodic  Speech
    # If you used full strings instead of codes, add them here:
    'Mon-Note'   : {'Melody':0, 'Speech':0},
    'Mel-Note'   : {'Melody':1, 'Speech':0},
    'Mon-Speech' : {'Melody':0, 'Speech':1},
    'Mel-Speech' : {'Melody':1, 'Speech':1},
}

###############################################################################
# 1) PREP: harmonize columns / add Melody & Speech
###############################################################################
def prepare_df(raw_df):
    df = raw_df.rename(columns={
        COLUMN_ALIASES['subject']  : 'Subject',
        COLUMN_ALIASES['song']     : 'Song',
        COLUMN_ALIASES['condition']: 'Condition',
        COLUMN_ALIASES['score']    : 'Score',
        COLUMN_ALIASES['feedback'] : 'Feedback'
    }).copy()

    # Ensure categorical types
    df['Subject']  = df['Subject'].astype(str)
    df['Song']     = df['Song'].astype(str)
    df['Condition'] = df['Condition'].astype(str)

    # Map Feedback to 0/1 if needed
    if not np.issubdtype(df['Feedback'].dtype, np.number):
        fb_map = {k:i for i,k in enumerate(sorted(df['Feedback'].unique()))}
        # try to prefer 'No'->0, 'Yes'->1 when possible
        if set(df['Feedback'].str.lower()) >= {'no','yes'}:
            fb_map = {'no':0,'yes':1}
            df['Feedback'] = df['Feedback'].str.lower().map(fb_map)
        else:
            df['Feedback'] = df['Feedback'].map(fb_map)
    df['Feedback'] = df['Feedback'].astype(int)

    # Add Melody / Speech
    mel = []
    spc = []
    for c in df['Condition']:
        c_key = c
        if c_key not in CONDITION_TO_FACTORS:
            # attempt uppercase no-dash
            c_key = c.replace('-', '').upper()
        if c_key not in CONDITION_TO_FACTORS:
            raise ValueError(f"Condition code '{c}' not found in CONDITION_TO_FACTORS mapping.")
        mel.append(CONDITION_TO_FACTORS[c_key]['Melody'])
        spc.append(CONDITION_TO_FACTORS[c_key]['Speech'])
    df['Melody'] = mel
    df['Speech'] = spc

    # For readability: categorical labels
    df['Melody_lab'] = df['Melody'].map({0:'Monotone', 1:'Melodic'})
    df['Speech_lab'] = df['Speech'].map({0:'Note', 1:'Speech'})
    df['FB_lab']     = df['Feedback'].map({0:'NoFB', 1:'FB'})

    return df

###############################################################################
# 2) FULL FACTORIAL ANOVA (Subject blocked)
###############################################################################
def run_full_anova(df, typ=2):
    """
    OLS with Subject as a fixed blocking factor.
    Equivalent to repeated-measures ANOVA on cell means when balanced.
    """
    # We treat Subject as categorical fixed effect; all within-subject.
    model = smf.ols(
        'Score ~ C(Subject) + C(Song)*C(Melody)*C(Speech)*C(Feedback)',
        data=df
    ).fit()
    aov = sm.stats.anova_lm(model, typ=typ)
    return aov, model

###############################################################################
# 3) UTILITIES FOR WITHIN-SUBJECT CONTRASTS
###############################################################################
def _pivot_subject_means(df, within, collapse_over=None):
    """
    Return pivot table of subject means over specified 'within' factors.
    collapse_over: iterable of factor names to average across (ignored if None).
    Example: within=['Speech','Melody'], collapse_over=['Song','Feedback']
    """
    cols = ['Subject'] + within
    if collapse_over is not None:
        cols += collapse_over
    tmp = df[cols + ['Score']].copy()
    # group across all columns except Score
    gb_cols = ['Subject'] + within
    tmp = tmp.groupby(gb_cols)['Score'].mean().reset_index()
    return tmp

def cohens_dz(diff_scores):
    """Cohen's dz for paired design."""
    return diff_scores.mean() / diff_scores.std(ddof=1)

def ci95_paired(diff_scores):
    n = len(diff_scores)
    m = diff_scores.mean()
    se = diff_scores.std(ddof=1) / np.sqrt(n)
    h = stats.t.ppf(0.975, df=n-1) * se
    return m - h, m + h

def paired_summary(x, y, label_x='x', label_y='y'):
    """
    Return dict with paired t-test results (y - x).
    """
    diff = y - x
    t, p = stats.ttest_rel(y, x)
    ci_lo, ci_hi = ci95_paired(diff)
    return {
        'n'      : len(diff),
        'mean_x' : x.mean(),
        'mean_y' : y.mean(),
        'mean_diff' : diff.mean(),
        'ci95_lo'   : ci_lo,
        'ci95_hi'   : ci_hi,
        't'      : t,
        'df'     : len(diff)-1,
        'p'      : p,
        'dz'     : cohens_dz(diff),
        'label_x': label_x,
        'label_y': label_y
    }

###############################################################################
# 4) SIMPLE EFFECTS: MELODY × SPEECH
###############################################################################
def simple_effect_melody_by_speech(df, collapse_over=('Song','Feedback')):
    """
    Compute:
      a) effect of Melody within Speech=0 (notes)
      b) effect of Melody within Speech=1 (speech)
      c) interaction (difference-of-differences)
    """
    tmp = _pivot_subject_means(df, within=['Speech','Melody'], collapse_over=collapse_over)
    # pivot Speech -> rows; Melody -> columns
    pvt = tmp.pivot_table(index='Subject', columns=['Speech','Melody'], values='Score')
    # columns: (0,0),(0,1),(1,0),(1,1)  i.e., Speech,Melody
    # Extract arrays
    Note_Mono = pvt[(0,0)].values
    Note_Melo = pvt[(0,1)].values
    Spch_Mono = pvt[(1,0)].values
    Spch_Melo = pvt[(1,1)].values

    # simple effects
    eff_note  = paired_summary(Note_Mono, Note_Melo, label_x='Mon-Note', label_y='Mel-Note')
    eff_spch  = paired_summary(Spch_Mono, Spch_Melo, label_x='Mon-Speech', label_y='Mel-Speech')

    # interaction: (Melody effect in Notes) vs (Melody effect in Speech)
    d_note  = Note_Melo - Note_Mono
    d_spch  = Spch_Melo - Spch_Mono
    t_int, p_int = stats.ttest_rel(d_note, d_spch)
    ci_lo, ci_hi = ci95_paired(d_note - d_spch)
    int_res = {
        'n'        : len(d_note),
        'mean_diff_note'  : d_note.mean(),
        'mean_diff_speech': d_spch.mean(),
        'mean_diffdiff'   : (d_note - d_spch).mean(),
        'ci95_lo'   : ci_lo,
        'ci95_hi'   : ci_hi,
        't'         : t_int,
        'df'        : len(d_note)-1,
        'p'         : p_int,
        'dz'        : cohens_dz(d_note - d_spch),
        'label'     : 'Melody×Speech interaction'
    }

    return eff_note, eff_spch, int_res

###############################################################################
# 5) SIMPLE EFFECTS: FEEDBACK × SPEECH
###############################################################################
def simple_effect_feedback_by_speech(df, collapse_over=('Song','Melody')):
    """
    Compute feedback simple effects within Speech levels and their interaction.
    """
    tmp = _pivot_subject_means(df, within=['Speech','Feedback'], collapse_over=collapse_over)
    pvt = tmp.pivot_table(index='Subject', columns=['Speech','Feedback'], values='Score')

    Note_NoFB = pvt[(0,0)].values
    Note_FB   = pvt[(0,1)].values
    Spch_NoFB = pvt[(1,0)].values
    Spch_FB   = pvt[(1,1)].values

    eff_note  = paired_summary(Note_NoFB, Note_FB, label_x='NoFB-Note', label_y='FB-Note')
    eff_spch  = paired_summary(Spch_NoFB, Spch_FB, label_x='NoFB-Speech', label_y='FB-Speech')

    d_note = Note_FB - Note_NoFB
    d_spch = Spch_FB - Spch_NoFB
    t_int, p_int = stats.ttest_rel(d_note, d_spch)
    ci_lo, ci_hi = ci95_paired(d_note - d_spch)
    int_res = {
        'n'        : len(d_note),
        'mean_diff_note'  : d_note.mean(),
        'mean_diff_speech': d_spch.mean(),
        'mean_diffdiff'   : (d_note - d_spch).mean(),
        'ci95_lo'   : ci_lo,
        'ci95_hi'   : ci_hi,
        't'         : t_int,
        'df'        : len(d_note)-1,
        'p'         : p_int,
        'dz'        : cohens_dz(d_note - d_spch),
        'label'     : 'Feedback×Speech interaction'
    }
    return eff_note, eff_spch, int_res

###############################################################################
# 6) OPTIONAL: break down interactions BY SONG
###############################################################################
def simple_effect_melody_by_speech_per_song(df):
    """
    Returns a dict keyed by song with (eff_note, eff_spch, int_res) for that song
    (collapsing only over Feedback).
    """
    out = {}
    for song, sdf in df.groupby('Song'):
        out[song] = simple_effect_melody_by_speech(sdf, collapse_over=('Feedback',))
    return out

def simple_effect_feedback_by_speech_per_song(df):
    out = {}
    for song, sdf in df.groupby('Song'):
        out[song] = simple_effect_feedback_by_speech(sdf, collapse_over=('Melody',))
    return out

###############################################################################
# 7) PRETTY PRINT
###############################################################################
def print_paired_summary(res, title):
    print(f"\n=== {title} ===")
    if isinstance(res, tuple):  # (eff1, eff2, int)
        eff1, eff2, intr = res
        for eff in (eff1, eff2):
            print(f"{eff['label_x']} vs {eff['label_y']}: "
                  f"mean {eff['label_x']:.>15s} = {eff['mean_x']:.3f}, "
                  f"mean {eff['label_y']:.>15s} = {eff['mean_y']:.3f}, "
                  f"Δ = {eff['mean_diff']:.3f} "
                  f"[{eff['ci95_lo']:.3f}, {eff['ci95_hi']:.3f}], "
                  f"t({eff['df']}) = {eff['t']:.2f}, p = {eff['p']:.4f}, dz = {eff['dz']:.2f}")
        print(f"Interaction ({intr['label']}): Δ_note = {intr['mean_diff_note']:.3f}, "
              f"Δ_speech = {intr['mean_diff_speech']:.3f}, "
              f"ΔΔ = {intr['mean_diffdiff']:.3f} "
              f"[{intr['ci95_lo']:.3f}, {intr['ci95_hi']:.3f}], "
              f"t({intr['df']}) = {intr['t']:.2f}, p = {intr['p']:.4f}, dz = {intr['dz']:.2f}")
    else:
        raise ValueError("Unexpected result format.")

###############################################################################
# 8) QUICK DEMO PIPELINE (call these in order)
###############################################################################
def run_all_stats(raw_df):
    df = prepare_df(raw_df)
    print("Data prepared. Factor levels:\n",
          df[['Melody_lab','Speech_lab','FB_lab','Song']].drop_duplicates().sort_values(['Melody_lab','Speech_lab','FB_lab','Song']))

    # Full ANOVA
    aov, model = run_full_anova(df)
    print("\n========== FULL FACTORIAL ANOVA (typ=2) ==========")
    print(aov)

    # Planned simple effects: Melody×Speech (collapsed over Song & Feedback)
    ms_res = simple_effect_melody_by_speech(df)
    print_paired_summary(ms_res, "Melody simple effects within Speech levels (collapsed over Song, Feedback)")

    # Planned simple effects: Feedback×Speech (collapsed over Song & Melody)
    fs_res = simple_effect_feedback_by_speech(df)
    print_paired_summary(fs_res, "Feedback simple effects within Speech levels (collapsed over Song, Melody)")

    # Optional by-song breakdowns
    print("\n========== BY SONG: Melody×Speech ==========")
    for song, res in simple_effect_melody_by_speech_per_song(df).items():
        print_paired_summary(res, f"Song: {song}")

    print("\n========== BY SONG: Feedback×Speech ==========")
    for song, res in simple_effect_feedback_by_speech_per_song(df).items():
        print_paired_summary(res, f"Song: {song}")

    return df, aov

###############################################################################
# 9) UPDATED PLOTTING (Song panel + 2×2 panel)
###############################################################################
import matplotlib.pyplot as plt

def plot_song_and_MxS(df):
    """
    Replaces your old combined_song_condition() plot.
    Panel A: Song (collapsed over Melody, Speech, Feedback)
    Panel B: 2×2 Melody×Speech combinations (collapsed over Song, Feedback)
    """
    # Panel A data
    song_pivot = (
        df.groupby(['Subject','Song'])['Score']
          .mean()
          .unstack('Song')
          .sort_index(axis=1)
    )

    # Panel B data (Melody x Speech)
    ms = (
        df.groupby(['Subject','Melody_lab','Speech_lab'])['Score']
          .mean()
          .reset_index()
    )
    ms['Combo'] = ms['Melody_lab'].str[0] + '-' + ms['Speech_lab']  # e.g., "M-Note" / "M-Speech"
    ms_pivot = ms.pivot_table(index='Subject', columns='Combo', values='Score')
    # reorder columns if present
    cols_order = [c for c in ['Monotone-Note','Melodic-Note','Monotone-Speech','Melodic-Speech'] if c in ms_pivot.columns]
    ms_pivot = ms_pivot[cols_order]

    fig, axes = plt.subplots(1, 2, figsize=(12,5), sharey=True)
    _boxscatter(axes[0], song_pivot, 'A) Error by Song')
    _boxscatter(axes[1], ms_pivot,   'B) Error by Melody × Speech')
    axes[0].set_ylabel('Tapping error (%)')
    plt.tight_layout()
    plt.show()

def _boxscatter(ax, pivot_df, title):
    bp = ax.boxplot(pivot_df.values, widths=0.8, showmeans=True, meanline=True)
    # Overplot subject points
    for i, col in enumerate(pivot_df.columns, start=1):
        ax.scatter(np.full(pivot_df[col].shape, i),
                   pivot_df[col].values, s=6)
    ax.set_xticks(range(1, len(pivot_df.columns)+1))
    ax.set_xticklabels(pivot_df.columns, rotation=30, ha='right')
    ax.set_title(title)

###############################################################################
# END MODULE
###############################################################################

# Example usage (uncomment when df is defined):
df_prep, aov_table = run_all_stats(df)
plot_song_and_MxS(df_prep)


In [ ]:
###############################################################################
# LIBRARIES
###############################################################################
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

###############################################################################
# USER SETTINGS  ––  adapte si tes noms de colonnes diffèrent
###############################################################################
COLUMN_ALIASES = {
    'subject'   : 'Subject',     # identifiant participant
    'song'      : 'Song',        # 'Balou' / 'Papaoutai' / 'Piaf'
    'condition' : 'Condition',   # codes: 'B','M','S','SM' ou libellés complets
    'score'     : 'Score',       # tapping‑error (%)
    'feedback'  : 'Feedback'     # 0/1 ou 'No'/'Yes'
}

CONDITION_TO_FACTORS = {
    'B'  : {'Melody':0, 'Speech':0},    # Monotone‑Note
    'M'  : {'Melody':1, 'Speech':0},    # Melodic‑Note
    'S'  : {'Melody':0, 'Speech':1},    # Monotone‑Speech
    'SM' : {'Melody':1, 'Speech':1},    # Melodic‑Speech
    'Mon-Note'   : {'Melody':0, 'Speech':0},
    'Mel-Note'   : {'Melody':1, 'Speech':0},
    'Mon-Speech' : {'Melody':0, 'Speech':1},
    'Mel-Speech' : {'Melody':1, 'Speech':1},
}

###############################################################################
# 1) PREPARE DATAFRAME
###############################################################################
def prepare_df(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Harmonise les noms de colonnes et ajoute les facteurs Melody / Speech."""
    df = raw_df.rename(columns={
        COLUMN_ALIASES['subject']  : 'Subject',
        COLUMN_ALIASES['song']     : 'Song',
        COLUMN_ALIASES['condition']: 'Condition',
        COLUMN_ALIASES['score']    : 'Score',
        COLUMN_ALIASES['feedback'] : 'Feedback'
    }).copy()

    # Types
    df['Subject']   = df['Subject'].astype(str)
    df['Song']      = df['Song'].astype(str)
    df['Condition'] = df['Condition'].astype(str)

    # Feedback en 0/1
    if not np.issubdtype(df['Feedback'].dtype, np.number):
        if set(df['Feedback'].str.lower()) >= {'no', 'yes'}:
            df['Feedback'] = df['Feedback'].str.lower().map({'no':0, 'yes':1})
        else:
            df['Feedback'] = df['Feedback'].astype('category').cat.codes
    df['Feedback'] = df['Feedback'].astype(int)

    # Ajout Melody / Speech
    mel, spc = [], []
    for c in df['Condition']:
        key = c.replace('-', '')
        if key not in CONDITION_TO_FACTORS:
            raise ValueError(f"Condition '{c}' non trouvée dans le mapping.")
        mel.append(CONDITION_TO_FACTORS[key]['Melody'])
        spc.append(CONDITION_TO_FACTORS[key]['Speech'])
    df['Melody'] = mel
    df['Speech'] = spc

    # Labels lisibles
    df['Melody_lab'] = df['Melody'].map({0:'Monotone', 1:'Melodic'})
    df['Speech_lab'] = df['Speech'].map({0:'Note', 1:'Speech'})
    df['FB_lab']     = df['Feedback'].map({0:'NoFB', 1:'FB'})
    return df

###############################################################################
# 2) FACTORIAL ANOVA –– MÉDIANES sujet × cellule
###############################################################################
def run_full_anova_median(df: pd.DataFrame, typ:int=2):
    """
    1) Pour chaque sujet et chaque cellule 3×2×2×2, calcule la médiane du Score.
    2) Applique une ANOVA OLS avec Sujet comme bloc fixe.
    """
    cell_medians = (df.groupby(['Subject','Song','Melody','Speech','Feedback'])
                      .agg({'Score':'median'})
                      .reset_index())

    model = smf.ols(
        'Score ~ C(Subject) + C(Song)*C(Melody)*C(Speech)*C(Feedback)',
        data=cell_medians
    ).fit()
    aov = sm.stats.anova_lm(model, typ=typ)
    return aov, model

###############################################################################
# 3) PAIREDS t + OUTILS
###############################################################################
def cohens_dz(diff):
    return diff.mean() / diff.std(ddof=1)

def ci95(diff):
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    h  = stats.t.ppf(0.975, len(diff)-1) * se
    return diff.mean()-h, diff.mean()+h

def paired_summary(x, y, label_x, label_y):
    diff = y - x
    t, p = stats.ttest_rel(y, x)
    lo, hi = ci95(diff)
    return dict(n=len(diff), mean_x=x.mean(), mean_y=y.mean(),
                diff=diff.mean(), ci_lo=lo, ci_hi=hi,
                t=t, df=len(diff)-1, p=p, dz=cohens_dz(diff),
                label_x=label_x, label_y=label_y)

###############################################################################
# 4) CONTRASTES  –– MÉDIANES
###############################################################################
def _subject_pivot(df, index_cols, collapse_over=None):
    """
    Renvoie un tableau Sujet × facteurs avec la médiane.
    index_cols : liste des facteurs d’intérêt
    collapse_over : itérable (ou None) de facteurs à moyenner
    """
    # ► assure que collapse_over est une liste
    collapse_over = list(collapse_over) if collapse_over else []

    cols = ['Subject'] + index_cols + collapse_over
    g = (df[cols + ['Score']]
           .groupby(['Subject'] + index_cols)['Score']
           .median()
           .reset_index())
    return g

def melody_by_speech(df, collapse_over=('Song','Feedback')):
    g = _subject_pivot(df, ['Speech','Melody'], collapse_over)
    pvt = g.pivot_table(index='Subject', columns=['Speech','Melody'],
                        values='Score')
    note_mono  = pvt[(0,0)].values
    note_melo  = pvt[(0,1)].values
    spch_mono  = pvt[(1,0)].values
    spch_melo  = pvt[(1,1)].values

    eff_note = paired_summary(note_mono, note_melo,
                              'Mon‑Note', 'Mel‑Note')
    eff_spch = paired_summary(spch_mono, spch_melo,
                              'Mon‑Speech', 'Mel‑Speech')
    diff_note = note_melo - note_mono
    diff_spch = spch_melo - spch_mono
    t_int, p_int = stats.ttest_rel(diff_note, diff_spch)
    lo, hi = ci95(diff_note - diff_spch)
    inter = dict(label='Melody×Speech', n=len(diff_note),
                 diff_note=diff_note.mean(), diff_spch=diff_spch.mean(),
                 diffdiff=(diff_note-diff_spch).mean(),
                 ci_lo=lo, ci_hi=hi, t=t_int, df=len(diff_note)-1,
                 p=p_int, dz=cohens_dz(diff_note-diff_spch))
    return eff_note, eff_spch, inter

def feedback_by_speech(df, collapse_over=('Song','Melody')):
    g = _subject_pivot(df, ['Speech','Feedback'], collapse_over)
    pvt = g.pivot_table(index='Subject', columns=['Speech','Feedback'],
                        values='Score')
    note_no   = pvt[(0,0)].values
    note_fb   = pvt[(0,1)].values
    spch_no   = pvt[(1,0)].values
    spch_fb   = pvt[(1,1)].values

    eff_note = paired_summary(note_no, note_fb, 'NoFB‑Note', 'FB‑Note')
    eff_spch = paired_summary(spch_no, spch_fb, 'NoFB‑Speech', 'FB‑Speech')
    diff_note = note_fb - note_no
    diff_spch = spch_fb - spch_no
    t_int, p_int = stats.ttest_rel(diff_note, diff_spch)
    lo, hi = ci95(diff_note - diff_spch)
    inter = dict(label='Feedback×Speech', n=len(diff_note),
                 diff_note=diff_note.mean(), diff_spch=diff_spch.mean(),
                 diffdiff=(diff_note-diff_spch).mean(),
                 ci_lo=lo, ci_hi=hi, t=t_int, df=len(diff_note)-1,
                 p=p_int, dz=cohens_dz(diff_note-diff_spch))
    return eff_note, eff_spch, inter

###############################################################################
# 5) AFFICHAGE CONTRASTES
###############################################################################
def print_contrast(res, title):
    a, b, inter = res
    print(f"\n=== {title} ===")
    for r in (a, b):
        print(f"{r['label_x']} vs {r['label_y']}: Δ = {r['diff']:.3f} "
              f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}], p = {r['p']:.4f}")
    print(f"Interaction {inter['label']}: ΔΔ = {inter['diffdiff']:.3f}, "
          f"p = {inter['p']:.4f}")

###############################################################################
# 6) PLOT (moyennes visuelles inchangées)
###############################################################################
def plot_song_and_MxS(df):
    song_pivot = (df.groupby(['Subject','Song'])['Score']
                    .median()                           # médiane per‑subject
                    .unstack('Song')
                    .sort_index(axis=1))

    ms = (df.groupby(['Subject','Melody_lab','Speech_lab'])['Score']
            .median()
            .reset_index())
    ms['Combo'] = ms['Melody_lab'].str[0] + '-' + ms['Speech_lab']
    ms_pivot = ms.pivot_table(index='Subject', columns='Combo', values='Score')
    order = [c for c in ['M-Note','M-Speech','M-Note','M-Speech'] if c in ms_pivot]
    ms_pivot = ms_pivot[order] if order else ms_pivot

    fig, ax = plt.subplots(1,2, figsize=(12,5), sharey=True)
    for i, (pvt, title) in enumerate(zip([song_pivot, ms_pivot],
                                         ['Song', 'Melody×Speech'])):
        ax[i].boxplot(pvt.values, showmeans=True, meanline=True)
        for j, col in enumerate(pvt.columns, 1):
            ax[i].scatter(np.full(pvt[col].shape, j), pvt[col], s=6)
        ax[i].set_xticks(range(1, len(pvt.columns)+1))
        ax[i].set_xticklabels(pvt.columns, rotation=30, ha='right')
        ax[i].set_title(title)
    ax[0].set_ylabel('Tapping error (%)')
    plt.tight_layout()
    plt.show()

###############################################################################
# 7) PIPELINE PRINCIPAL
###############################################################################
def run_all_stats_median(raw_df):
    df = prepare_df(raw_df)

    print("Data prepared. Factor levels:")
    print(df[['Melody_lab','Speech_lab','FB_lab','Song']].drop_duplicates()
             .sort_values(['Melody_lab','Speech_lab','FB_lab','Song']))

    # ANOVA
    aov, _ = run_full_anova_median(df)
    print("\n========== MEDIAN‑BASED ANOVA ==========")
    print(aov)

    # Contrastes planifiés
    print_contrast(melody_by_speech(df),   "Melody × Speech (médiane)")
    print_contrast(feedback_by_speech(df), "Feedback × Speech (médiane)")

    return df, aov

# utilise ton objet existant
df_prep, aov_med = run_all_stats_median(df)
plot_song_and_MxS(df_prep)


In [ ]:
group_definitions = [
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['M', 'SM'],
        'feedback': 'Both',
        'name': 'Melodic'
    },
    {
        'songs': ['Balou', 'Piaf', 'Papaoutai'],
        'conditions': ['B', 'S'],
        'feedback': 'Both',  
        'name': 'Not Melodic'
    },
]

analyze_groups(df, group_definitions, outliers=True)